<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_07_model_evaluation/stage_07_model_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_07 – Model Evaluation**


# **Preparación de entorno**

## **1. Imports**

In [2]:
# Permite anotaciones modernas en Python < 3.11
from __future__ import annotations

# ================================
# Standard library
# ================================
import os
import sys
import json
import time
import random
import importlib
import warnings
import logging
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Iterable, Tuple

# ================================
# Third-party
# ================================
import numpy as np
import joblib

# ================================
# Configuración global
# ================================

# ---- Warnings (controlado, no agresivo)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ---- Logging limpio para notebooks
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
logger.propagate = False  # evita duplicación con el root logger

# limpiar handlers si se re-ejecuta la celda
if logger.handlers:
    logger.handlers.clear()

handler = logging.StreamHandler()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
handler.setFormatter(formatter)
logger.addHandler(handler)

logger.info("Environment initialized")

2026-04-23 00:46:02,540 | INFO | Environment initialized


## **2. Acceso a drive**

In [3]:
# ================================
# Entorno (Google Drive / local)
# ================================

from pathlib import Path
import os

# Detectar si estamos en Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    DEFAULT_DRIVE_DIR = "/content/drive/MyDrive/neural_profit/"
else:
    # fallback local (puedes ajustarlo si quieres)
    DEFAULT_DRIVE_DIR = "./neural_profit/"

# Ruta base del proyecto
DRIVE_DIR = Path(os.environ.get("DRIVE_DIR", DEFAULT_DRIVE_DIR))

# Validación básica
if not DRIVE_DIR.exists():
    logger.warning(f"DRIVE_DIR no existe: {DRIVE_DIR}")
else:
    logger.info(f"DRIVE_DIR: {DRIVE_DIR}")

2026-04-23 00:46:21,892 | INFO | DRIVE_DIR: /content/drive/MyDrive/neural_profit


Mounted at /content/drive


## **3. Carga de métricas y probabilidades**

In [9]:
from pathlib import Path
import pandas as pd

def load_all_model_results(
    *,
    base_dir: Path,
    split: str = "valid",
    verbose: bool = True,
) -> dict[str, dict[str, pd.DataFrame]]:
    """
    Carga métricas de clasificación y salidas probabilísticas de todos los modelos.

    Estructura esperada:
        base_dir/
            classification_metrics/
                classification_metrics_<model>_<split>.parquet
            classification_probabilities/
                classification_probabilities_<model>_<split>.parquet

    Retorna:
        {
            "metrics": {
                "gru": df,
                "logistic_regression": df,
                ...
            },
            "probabilities": {
                "gru": df,
                "logistic_regression": df,
                ...
            }
        }
    """

    metrics_dir = base_dir / "classification_metrics"
    probabilities_dir = base_dir / "classification_probabilities"

    results = {
        "metrics": {},
        "probabilities": {},
    }

    # =========================
    # Cargar métricas
    # =========================
    metrics_pattern = f"classification_metrics_*_{split}.parquet"
    metric_paths = sorted(metrics_dir.glob(metrics_pattern))

    if verbose:
        print("=" * 90)
        print(f"[INFO] Métricas encontradas: {len(metric_paths)}")
        print("=" * 90)

    for path in metric_paths:
        # Ejemplo:
        # classification_metrics_gru_valid.parquet
        model_name = path.stem.replace("classification_metrics_", "").replace(f"_{split}", "")

        if verbose:
            print(f"[LOAD][METRICS] {model_name} -> {path}")

        df = pd.read_parquet(path)
        results["metrics"][model_name] = df

    # =========================
    # Cargar probabilidades
    # =========================
    prob_pattern = f"classification_probabilities_*_{split}.parquet"
    prob_paths = sorted(probabilities_dir.glob(prob_pattern))

    if verbose:
        print("=" * 90)
        print(f"[INFO] Probabilidades encontradas: {len(prob_paths)}")
        print("=" * 90)

    for path in prob_paths:
        # Ejemplo:
        # classification_probabilities_gru_valid.parquet
        model_name = path.stem.replace("classification_probabilities_", "").replace(f"_{split}", "")

        if verbose:
            print(f"[LOAD][PROBA] {model_name} -> {path}")

        df = pd.read_parquet(path)
        results["probabilities"][model_name] = df

    return results

In [10]:
metrics_base_dir = DRIVE_DIR / "metrics"

all_results = load_all_model_results(
    base_dir=metrics_base_dir,
    split="valid",
    verbose=True,
)

metrics_dict = all_results["metrics"]
probabilities_dict = all_results["probabilities"]

[INFO] Métricas encontradas: 5
[LOAD][METRICS] gru -> /content/drive/MyDrive/neural_profit/metrics/classification_metrics/classification_metrics_gru_valid.parquet
[LOAD][METRICS] logistic_regression -> /content/drive/MyDrive/neural_profit/metrics/classification_metrics/classification_metrics_logistic_regression_valid.parquet
[LOAD][METRICS] random_forest -> /content/drive/MyDrive/neural_profit/metrics/classification_metrics/classification_metrics_random_forest_valid.parquet
[LOAD][METRICS] transformer -> /content/drive/MyDrive/neural_profit/metrics/classification_metrics/classification_metrics_transformer_valid.parquet
[LOAD][METRICS] xgboost -> /content/drive/MyDrive/neural_profit/metrics/classification_metrics/classification_metrics_xgboost_valid.parquet
[INFO] Probabilidades encontradas: 5
[LOAD][PROBA] gru -> /content/drive/MyDrive/neural_profit/metrics/classification_probabilities/classification_probabilities_gru_valid.parquet
[LOAD][PROBA] logistic_regression -> /content/drive/My

In [11]:
models_metrics = set(metrics_dict.keys())
models_prob = set(probabilities_dict.keys())

common_models = sorted(models_metrics & models_prob)
only_metrics = sorted(models_metrics - models_prob)
only_prob = sorted(models_prob - models_metrics)

print("Modelos con ambos archivos:", common_models)
print("Solo métricas:", only_metrics)
print("Solo probabilidades:", only_prob)

Modelos con ambos archivos: ['gru', 'logistic_regression', 'random_forest', 'transformer', 'xgboost']
Solo métricas: []
Solo probabilidades: []


# PLAN DE SELECCIÓN Y TUNING (SEQ2ONE T2)


Claro, Gus. Te dejo el bloque en markdown listo para pegar en la notebook, limpio y formal.

---

1. Consolidación de resultados

Se mantiene la estructura actual de consolidación, asegurando que todos los modelos se evalúan bajo condiciones comparables. El foco principal está en la consistencia entre modelos, la integridad del dataset y la comparabilidad real de los resultados. Esta etapa no requiere modificaciones relevantes respecto a la versión original.

---

2. Definición de métricas de decisión

Antes de iniciar el análisis, se deben definir explícitamente las métricas que guiarán la selección de modelos.

Métricas principales de ranking:

* balanced_accuracy (métrica principal)
* f1_macro

Métricas secundarias:

* accuracy
* métricas de mejora respecto al baseline naive

Métricas operativas (derivadas de probabilidades):

* trade_rate
* precision_useful
* confidence

Esta definición es crítica, ya que establece el criterio objetivo para comparar modelos.

---

3. Análisis de probabilidades

Se incorpora un análisis específico de las salidas probabilísticas de los modelos antes de aplicar cualquier filtro.

Se evalúa:

* distribución de confidence
* distribución de clases predichas
* sesgo hacia la clase neutra (0)
* calibración implícita de probabilidades

Este análisis permite detectar problemas típicos como modelos que no generan señales, modelos colapsados a una sola clase o presencia de sobreconfianza en las predicciones.

---

4. Filtro de calidad (anti-ruido)

Se mantiene el concepto de filtrado, pero se formalizan criterios explícitos para eliminar modelos no operativos.

Ejemplo de criterios:

* trade_rate > 0.05
* precision_useful superior al baseline
* mejora positiva en balanced_accuracy respecto al naive

Este filtro elimina modelos que presentan buenas métricas teóricas pero carecen de utilidad práctica.

---

5. Ranking por modelo (nivel arquitectura)

Se agrupan los resultados por tipo de modelo (arquitectura) y se comparan en términos de:

* promedio de métricas
* estabilidad entre distintos targets

Este análisis permite identificar qué tipo de modelo (por ejemplo, GRU, Transformer, XGBoost o regresión logística) presenta mejor comportamiento general.

---

6. Análisis por window_size (L)

Se evalúa el impacto del tamaño de ventana en el desempeño del modelo.

Se analiza:

* variación de métricas según L
* posibles efectos de sobreajuste temporal
* pérdida de señal con ventanas inadecuadas

Este punto es especialmente relevante en modelos secuenciales.

---

7. Análisis por target

Se analizan los distintos targets definidos (por ejemplo, t2_p40_h30, t2_p40_h60, etc.).

Se evalúa:

* calidad predictiva
* trade_rate
* estabilidad de resultados

Este análisis permite entender qué tipo de comportamiento del mercado está siendo capturado por el modelo.

---

8. Análisis conjunto (modelo + window_size + target)

Se realiza un análisis combinado considerando simultáneamente:

* modelo
* window_size
* target

Se construye un ranking de combinaciones en función de:

* métricas de clasificación
* métricas operativas

Este constituye el núcleo del proceso de selección.

---

9. Selección de candidatos

No se selecciona un único modelo, sino un conjunto reducido de candidatos.

Se eligen las mejores combinaciones (por ejemplo, top 3 a 5) considerando:

* performance
* estabilidad
* equilibrio entre frecuencia de señal y calidad

---

10. Validación operativa

Se utiliza el dataset de probabilidades para evaluar el comportamiento en condiciones cercanas a la operativa real.

Se aplica una regla de decisión basada en umbrales y se mide:

* número de trades por período
* cantidad de señales útiles
* distribución temporal de las señales

Este paso conecta la evaluación del modelo con su uso en un entorno de trading.

---

11. Definición del espacio de tuning

Una vez seleccionados los candidatos, se define el espacio de optimización.

Se consideran:

* hiperparámetros del modelo
* umbrales de decisión
* posibles mejoras en features

---

Resumen

La estructura original del análisis es adecuada, pero se refuerza mediante tres elementos clave:

* definición explícita de métricas de decisión
* incorporación del análisis de probabilidades
* validación operativa de los modelos

---

Conclusión

Se mantiene la estructura general del análisis, incorporando ajustes que permiten evaluar los modelos no solo desde el punto de vista predictivo, sino también como sistemas de generación de señales para trading.


# **1. Consolidación de resultados**

En esta etapa se unifican los resultados generados por todos los modelos, con el objetivo de asegurar consistencia, integridad y comparabilidad en el análisis posterior. La consolidación permite trabajar sobre estructuras homogéneas, facilitando la validación y el análisis conjunto de métricas y salidas probabilísticas.

Como primer paso, se realiza la concatenación de los resultados individuales de cada modelo en tablas unificadas. En particular:

* Se construye un único DataFrame (`df_metrics_all`) que contiene todas las métricas de clasificación.
* Se construye un único DataFrame (`df_probabilities_all`) que contiene todas las probabilidades predichas.

Ambas tablas incluyen una columna identificadora del modelo, lo que permite realizar análisis comparativos de forma directa sobre una estructura común.


## 1.0. **Concatenar métricas y probabilidades**

In [ ]:
def build_df_metrics_all(metrics_dict: dict[str, pd.DataFrame]) -> pd.DataFrame:
    dfs = []

    for model_name, df in metrics_dict.items():
        df_copy = df.copy()

        # Asegurar columna model
        if "model" not in df_copy.columns:
            df_copy["model"] = model_name
        else:
            df_copy["model"] = model_name  # sobrescribe por consistencia

        dfs.append(df_copy)

    df_metrics_all = pd.concat(dfs, axis=0, ignore_index=True)

    return df_metrics_all

In [ ]:
def build_df_probabilities_all(probabilities_dict: dict[str, pd.DataFrame]) -> pd.DataFrame:
    dfs = []

    for model_name, df in probabilities_dict.items():
        df_copy = df.copy()

        # Asegurar columna model
        if "model" not in df_copy.columns:
            df_copy["model"] = model_name
        else:
            df_copy["model"] = model_name

        dfs.append(df_copy)

    df_probabilities_all = pd.concat(dfs, axis=0, ignore_index=True)

    return df_probabilities_all

In [24]:
df_metrics = build_df_metrics_all(metrics_dict)
df_proba = build_df_probabilities_all(probabilities_dict)

(15, 58)
(103230, 49)


## **1.1. Verificación estructural**

In [27]:
print('1. Validación estructural:\n')
print('Métricas:\n')
df_metrics.info()
display(df_metrics.head())

print('\nProbabilidades:\n')
df_proba.info()
df_proba.head()

1. Validación estructural:

Métricas:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 58 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   model                            15 non-null     object 
 1   split                            15 non-null     object 
 2   window_size                      15 non-null     int64  
 3   target                           15 non-null     object 
 4   horizon                          15 non-null     int64  
 5   n_samples                        15 non-null     int64  
 6   accuracy                         15 non-null     float64
 7   balanced_accuracy                15 non-null     float64
 8   f1_macro                         15 non-null     float64
 9   f1_weighted                      15 non-null     float64
 10  precision_macro                  15 non-null     float64
 11  precision_weighted               15 non-null   

,model,split,window_size,target,horizon,n_samples,accuracy,balanced_accuracy,f1_macro,f1_weighted,...,min_samples_leaf,max_features,n_jobs,subsample,colsample_bytree,min_child_weight,gamma,reg_alpha,reg_lambda,tree_method
0,gru,valid,30,t2_p40_h30,30,6882,0.390003,0.415750,0.381825,0.372222,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,gru,valid,30,t2_p40_h60,60,6882,0.426039,0.417276,0.410322,0.414768,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,gru,valid,30,t2_p50_h30,30,6882,0.411799,0.402873,0.385423,0.389905,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,logistic_regression,valid,30,t2_p40_h30,30,6882,0.374164,0.396396,0.366836,0.360868,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,logistic_regression,valid,30,t2_p40_h60,60,6882,0.407004,0.395298,0.381194,0.387120,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Probabilidades:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103230 entries, 0 to 103229
Data columns (total 49 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   proba_-1           103230 non-null  float64
 1   proba_0            103230 non-null  float64
 2   proba_1            103230 non-null  float64
 3   pred_label         103230 non-null  int64  
 4   confidence         103230 non-null  float64
 5   y_true             103230 non-null  int8   
 6   is_correct         103230 non-null  bool   
 7   signal_raw         103230 non-null  int64  
 8   trade              103230 non-null  bool   
 9   model              103230 non-null  object 
 10  split              82584 non-null   object 
 11  window_size        82584 non-null   float64
 12  target             103230 non-null  object 
 13  horizon            103230 non-null  int64  
 14  class_weight_mode  103230 non-null  object 
 15  input_mode         82584 non-null

,proba_-1,proba_0,proba_1,pred_label,confidence,y_true,is_correct,signal_raw,trade,model,...,min_samples_leaf,max_features,n_jobs,subsample,colsample_bytree,min_child_weight,gamma,reg_alpha,reg_lambda,tree_method
0,0.232661,0.498282,0.269058,0,0.498282,1,False,0,False,gru,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.238767,0.492371,0.268861,0,0.492371,1,False,0,False,gru,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.242774,0.489684,0.267542,0,0.489684,1,False,0,False,gru,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.246056,0.489471,0.264472,0,0.489471,1,False,0,False,gru,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.243895,0.496987,0.259118,0,0.496987,1,False,0,False,gru,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Métricas (df_metrics)**

* No se observan valores nulos en las métricas principales.
* La estructura es consistente entre todos los modelos (mismas columnas).
* La cantidad de filas es coherente con el experimento (5 modelos × 3 targets = 15 filas).
* Existen muchas columnas con valores nulos asociadas a hiperparámetros. Esto es esperable, ya que cada modelo utiliza configuraciones distintas. Estas columnas no deben considerarse en análisis globales.
* Las columnas clave (`model`, `target`, `window_size`, `horizon`) están correctamente definidas.

Conclusión: el dataset de métricas es consistente y está listo para análisis comparativo.

---

**Probabilidades (df_proba)**

* El dataset tiene un volumen adecuado (~103k filas) y está completo en variables principales.
* Las columnas clave (`proba_*`, `pred_label`, `confidence`, `y_true`, `model`) están correctamente definidas.
* Se observan valores nulos en algunas columnas de metadata (`split`, `window_size`, `input_mode`). Esto es esperable y probablemente se debe a diferencias entre modelos (por ejemplo, modelos no secuenciales).
* Las variables operativas (`trade`, `signal_raw`, `is_correct`) ya están calculadas, lo cual facilita el análisis posterior.

Conclusión: el dataset de probabilidades es consistente y apto para análisis operativo.

---

**Observación general**

* El dataset de métricas opera a nivel agregado (una fila por combinación modelo–target).
* El dataset de probabilidades opera a nivel de observación (una fila por predicción individual).

Esta estructura es adecuada para realizar análisis comparativo de modelos y evaluar su comportamiento operativo.

---

**Conclusión general**

* No se detectan problemas estructurales.
* Los valores nulos observados son esperables y no representan un inconveniente.
* Ambos datasets están correctamente preparados para avanzar a las siguientes etapas del análisis.


## 1.2. Verificación columnas clave

En esta etapa se valida que los datasets consolidados contengan todas las columnas necesarias para el análisis.

Se define un conjunto mínimo de columnas esperadas para cada dataset:

- En métricas: columnas de identificación, métricas principales y comparación contra baseline
- En probabilidades: columnas de predicción, probabilidades, valores reales y variables operativas

El objetivo es detectar errores estructurales tempranos, como columnas faltantes o inconsistencias en la construcción de los datasets.

Si alguna columna clave no está presente, el análisis posterior puede ser inválido o incompleto.

In [28]:
print('2. Verificación columnas clave:\n')

# =========================================
# Métricas
# =========================================
cols_metrics_expected = [
    "model",
    "split",
    "window_size",
    "target",
    "horizon",
    "n_samples",
    "balanced_accuracy",
    "f1_macro",
    "accuracy",
    "balanced_accuracy_naive",
    "balanced_accuracy_gain_vs_naive",
    "class_weight_mode"
]

missing_metrics = [c for c in cols_metrics_expected if c not in df_metrics.columns]

print("Métricas - columnas faltantes:", missing_metrics)

# =========================================
# Probabilidades
# =========================================
cols_proba_expected = [
    "model",
    "split",
    "window_size",
    "target",
    "horizon",
    "proba_-1",
    "proba_0",
    "proba_1",
    "pred_label",
    "confidence",
    "y_true",
    "is_correct",
    "trade"
]

missing_proba = [c for c in cols_proba_expected if c not in df_proba.columns]

print("Probabilidades - columnas faltantes:", missing_proba)

# =========================================
# Chequeo adicional
# =========================================
print('\nChequeo general:')

print("Cantidad columnas df_metrics:", len(df_metrics.columns))
print("Cantidad columnas df_proba:", len(df_proba.columns))

2. Verificación columnas clave:

Métricas - columnas faltantes: []
Probabilidades - columnas faltantes: []

Chequeo general:
Cantidad columnas df_metrics: 58
Cantidad columnas df_proba: 49


* No se detectan columnas faltantes en ninguno de los datasets.
* Tanto `df_metrics` como `df_proba` contienen todas las variables necesarias para el análisis.
* La cantidad de columnas es elevada (58 en métricas, 49 en probabilidades), lo cual es esperable debido a la inclusión de hiperparámetros de distintos modelos.
* La presencia de estas columnas adicionales no representa un problema, pero deberán ser ignoradas en análisis globales y utilizadas solo cuando corresponda a cada modelo específico.

**Conclusión**

La estructura de ambos datasets es correcta y completa, por lo que se puede avanzar con confianza a las siguientes etapas del análisis.


## 1.3. Validar splits

Dado que el análisis se realiza únicamente sobre el conjunto de validación, se verifica que todas las observaciones pertenezcan efectivamente a dicho split. Este control permite detectar inconsistencias en la construcción del dataset, como mezclas involuntarias de distintos conjuntos.

In [29]:
print('3. Verificación split:\n')

print("df_metrics:")
print(df_metrics["split"].value_counts(dropna=False))

print("\ndf_proba:")
print(df_proba["split"].value_counts(dropna=False))

3. Verificación split:

df_metrics:
split
valid    15
Name: count, dtype: int64

df_proba:
split
valid    82584
NaN      20646
Name: count, dtype: int64


**Resultado de la verificación de split**

* En `df_metrics`, todas las observaciones pertenecen correctamente al split `valid`.
* En `df_proba`, la mayoría de las observaciones corresponden a `valid`, pero existe un conjunto relevante de filas con valores nulos en la columna `split`.

**Interpretación**

* La consistencia en `df_metrics` indica que las métricas fueron correctamente generadas sobre el conjunto de validación.
* Los valores nulos en `df_proba` son esperables y probablemente corresponden a modelos o pipelines donde esta metadata no fue incorporada explícitamente (por ejemplo, diferencias entre modelos tabulares y secuenciales).

**Implicancia para el análisis**

* No representa un problema estructural crítico, siempre que el dataset completo corresponda efectivamente a validación.
* Sin embargo, se pierde capacidad de filtrado por `split` en el dataset de probabilidades.

**Conclusión**

El dataset es consistente para el propósito actual (análisis sobre validación), pero se recomienda estandarizar la columna `split` en futuras versiones para mantener trazabilidad completa entre modelos.


## 1.4. Validar targets

Validación de targets

En esta etapa se verifica que los targets utilizados en el entrenamiento estén correctamente representados en los datasets consolidados.

El objetivo es asegurar que:

* todos los targets esperados están presentes
* la distribución es consistente entre modelos
* no existen errores de construcción (targets faltantes o mal nombrados)

Esto permite garantizar que la comparación entre modelos se realiza sobre el mismo conjunto de problemas.


In [30]:
print('4. Validación de targets:\n')

print("df_metrics:")
print(df_metrics["target"].value_counts())

print("\ndf_proba:")
print(df_proba["target"].value_counts())

4. Validación de targets:

df_metrics:
target
t2_p40_h30    5
t2_p40_h60    5
t2_p50_h30    5
Name: count, dtype: int64

df_proba:
target
t2_p40_h30    34410
t2_p40_h60    34410
t2_p50_h30    34410
Name: count, dtype: int64


Observaciones de la validación de targets

* Los tres targets están correctamente presentes en ambos datasets.
* En `df_metrics`, cada target tiene exactamente 5 registros → uno por modelo.
* En `df_proba`, todos los targets tienen la misma cantidad de observaciones → dataset balanceado.
* No hay targets faltantes ni inconsistencias en nombres.

Conclusión

La estructura de targets es consistente y permite una comparación justa entre modelos.


## 1.5. Validar ventanas (window_size)

En esta etapa se verifica que el tamaño de ventana utilizado en la construcción de los datos sea consistente en todo el dataset.

Dado que el experimento se realizó con un único valor de ventana (L = 30), el objetivo es confirmar que no existan mezclas de configuraciones que puedan invalidar la comparabilidad de los resultados.

In [31]:
print('5. Validación de window_size:\n')

print("df_metrics:")
print(sorted(df_metrics["window_size"].dropna().unique()))

print("\ndf_proba:")
print(sorted(df_proba["window_size"].dropna().unique()))

5. Validación de window_size:

df_metrics:
[np.int64(30)]

df_proba:
[np.float64(30.0)]


## 1.6. Validar duplicados

En esta etapa se verifica que no existan registros duplicados en los datasets, considerando las columnas que identifican de forma única cada experimento.

El objetivo es asegurar que cada combinación de modelo, target y configuración aparezca una única vez en el dataset de métricas, evitando sesgos en el análisis.

En el caso de probabilidades, no se espera unicidad total por fila (ya que cada fila es una observación), pero sí se puede validar duplicados exactos si fuera necesario.

In [32]:
print('6. Validación de duplicados:\n')

dup_cols = ["model", "split", "window_size", "target"]

dup_metrics = df_metrics.duplicated(subset=dup_cols).sum()

print("Duplicados en df_metrics:", dup_metrics)

6. Validación de duplicados:

Duplicados en df_metrics: 0


In [33]:
dup_proba = df_proba.duplicated().sum()

print("Duplicados exactos en df_proba:", dup_proba)

Duplicados exactos en df_proba: 3932


**Resultado de la validación de duplicados**

En df_metrics no se detectan duplicados. Esto confirma que cada combinación model + split + window_size + target aparece una sola vez, como corresponde.
En df_proba sí aparecen 3932 filas duplicadas exactas. Esto merece revisión, porque en principio cada fila debería representar una predicción individual.

**Interpretación**

En df_metrics, el resultado es correcto y esperado.
En df_proba, la presencia de duplicados puede deberse a dos causas principales:
concatenación repetida de alguna parte del dataset
observaciones realmente idénticas en todas sus columnas

**Implicancia para el análisis**

No es grave para la validación estructural inicial, pero no conviene ignorarlo.
Si estos duplicados provienen de una concatenación errónea, podrían sesgar análisis posteriores, especialmente en métricas operativas como trade_rate, confidence o distribución de señales.

**Conclusión**

El dataset de métricas está correcto. El dataset de probabilidades requiere una revisión puntual de los duplicados antes de avanzar con análisis operativos más finos.



### **1. Código sugerido para inspeccionarlos**

In [36]:
print("Duplicados exactos en df_proba:", df_proba.duplicated().sum())

df_proba_dups = df_proba[df_proba.duplicated(keep=False)].copy()

print("Shape duplicados:", df_proba_dups.shape)
display(df_proba_dups)

Duplicados exactos en df_proba: 3932
Shape duplicados: (7157, 49)


,proba_-1,proba_0,proba_1,pred_label,confidence,y_true,is_correct,signal_raw,trade,model,...,min_samples_leaf,max_features,n_jobs,subsample,colsample_bytree,min_child_weight,gamma,reg_alpha,reg_lambda,tree_method
41297,0.095,0.690,0.215,0,0.690,1,False,0,False,random_forest,...,1.0,sqrt,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
41299,0.095,0.680,0.225,0,0.680,1,False,0,False,random_forest,...,1.0,sqrt,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
41305,0.185,0.545,0.270,0,0.545,0,True,0,False,random_forest,...,1.0,sqrt,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
41314,0.145,0.610,0.245,0,0.610,0,True,0,False,random_forest,...,1.0,sqrt,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
41317,0.195,0.550,0.255,0,0.550,-1,False,0,False,random_forest,...,1.0,sqrt,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61927,0.310,0.440,0.250,0,0.440,1,False,0,False,random_forest,...,1.0,sqrt,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
61929,0.260,0.525,0.215,0,0.525,0,True,0,False,random_forest,...,1.0,sqrt,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
61930,0.250,0.450,0.300,0,0.450,0,True,0,False,random_forest,...,1.0,sqrt,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
61933,0.310,0.425,0.265,0,0.425,0,True,0,False,random_forest,...,1.0,sqrt,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [35]:
df_proba_nodup = df_proba.drop_duplicates().copy()

print("Shape original:", df_proba.shape)
print("Shape sin duplicados:", df_proba_nodup.shape)
print("Filas removidas:", df_proba.shape[0] - df_proba_nodup.shape[0])

Shape original: (103230, 49)
Shape sin duplicados: (99298, 49)
Filas removidas: 3932


### **2. Diágnostico completos de duplicados**

In [39]:
print("7. Diagnóstico de duplicados en df_proba:\n")

# =========================================
# 1. Duplicados exactos totales
# =========================================
n_dups = df_proba.duplicated().sum()

print(f"Duplicados exactos totales: {n_dups}")

# =========================================
# 2. Duplicados por modelo
# =========================================
print("\nDuplicados por modelo:")
dups_by_model = (
    df_proba[df_proba.duplicated()]
    .groupby("model")
    .size()
    .sort_values(ascending=False)
)
display(dups_by_model)

# =========================================
# 3. Duplicados por modelo y target
# =========================================
print("\nDuplicados por modelo y target:")
dups_by_model_target = (
    df_proba[df_proba.duplicated()]
    .groupby(["model", "target"])
    .size()
    .sort_values(ascending=False)
)
display(dups_by_model_target)

# =========================================
# 4. Comparación de tamaños antes y después
#    de eliminar duplicados
# =========================================
print("\nCantidad de filas por modelo y target (dataset original):")
size_original = (
    df_proba
    .groupby(["model", "target"])
    .size()
    .sort_values(ascending=False)
)
display(size_original)

print("\nCantidad de filas por modelo y target (sin duplicados):")
size_nodup = (
    df_proba
    .drop_duplicates()
    .groupby(["model", "target"])
    .size()
    .sort_values(ascending=False)
)
display(size_nodup)

# =========================================
# 5. Diferencia entre original y sin duplicados
# =========================================
print("\nDiferencia de filas removidas por modelo y target:")
diff_sizes = (size_original - size_nodup).sort_values(ascending=False)
display(diff_sizes[diff_sizes > 0])

# =========================================
# 6. Mostrar ejemplos reales de duplicados
#    ordenados para ver pares iguales juntos
# =========================================
print("\nEjemplos de filas duplicadas exactas (ordenadas):")
cols_compare = df_proba.columns.tolist()

df_dups_sorted = (
    df_proba[df_proba.duplicated(keep=False)]
    .sort_values(by=cols_compare)
)

display(df_dups_sorted.head(20))

7. Diagnóstico de duplicados en df_proba:

Duplicados exactos totales: 3932

Duplicados por modelo:


,0
model,
random_forest,3932



Duplicados por modelo y target:


model          target    
random_forest  t2_p50_h30    1362
               t2_p40_h30    1344
               t2_p40_h60    1226
dtype: int64


Cantidad de filas por modelo y target (dataset original):


...

model                target    
gru                  t2_p40_h30    6882
                     t2_p40_h60    6882
                     t2_p50_h30    6882
logistic_regression  t2_p40_h30    6882
                     t2_p40_h60    6882
                     t2_p50_h30    6882
transformer          t2_p40_h60    6882
xgboost              t2_p40_h60    6882
transformer          t2_p40_h30    6882
xgboost              t2_p40_h30    6882
transformer          t2_p50_h30    6882
xgboost              t2_p50_h30    6882
random_forest        t2_p40_h60    5656
                     t2_p40_h30    5538
                     t2_p50_h30    5520
dtype: int64


Diferencia de filas removidas por modelo y target:


model          target    
random_forest  t2_p50_h30    1362
               t2_p40_h30    1344
               t2_p40_h60    1226
dtype: int64


Ejemplos de filas duplicadas exactas (ordenadas):


,proba_-1,proba_0,proba_1,pred_label,confidence,y_true,is_correct,signal_raw,trade,model,...,min_samples_leaf,max_features,n_jobs,subsample,colsample_bytree,min_child_weight,gamma,reg_alpha,reg_lambda,tree_method
48290,0.025,0.950,0.025,0,0.950,0,True,0,False,random_forest,...,1.0,sqrt,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
48291,0.025,0.950,0.025,0,0.950,0,True,0,False,random_forest,...,1.0,sqrt,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
55061,0.040,0.845,0.115,0,0.845,1,False,0,False,random_forest,...,1.0,sqrt,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
55064,0.040,0.845,0.115,0,0.845,1,False,0,False,random_forest,...,1.0,sqrt,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
55172,0.045,0.865,0.090,0,0.865,0,True,0,False,random_forest,...,1.0,sqrt,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
55173,0.045,0.865,0.090,0,0.865,0,True,0,False,random_forest,...,1.0,sqrt,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
48295,0.050,0.890,0.060,0,0.890,0,True,0,False,random_forest,...,1.0,sqrt,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
51496,0.050,0.890,0.060,0,0.890,0,True,0,False,random_forest,...,1.0,sqrt,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
54788,0.055,0.420,0.525,1,0.525,1,True,1,True,random_forest,...,1.0,sqrt,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
54802,0.055,0.420,0.525,1,0.525,1,True,1,True,random_forest,...,1.0,sqrt,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### **3. Observaciones del diagnóstico de duplicados**



Se detectaron duplicados exactos únicamente en el modelo `random_forest`, afectando a los tres targets analizados. No se trata de una duplicación completa del dataset, sino de una repetición parcial de filas dentro del mismo modelo.

Esto implica que algunas observaciones aparecen más de una vez (en ciertos casos incluso dos o tres veces), mientras que otras aparecen una sola vez. Como resultado, el número de registros únicos para `random_forest` es menor al esperado, lo que confirma que la duplicación no es uniforme.

Este comportamiento no corresponde a un error global de concatenación, sino que indica una duplicación parcial originada probablemente en el pipeline específico de generación de predicciones de este modelo (por ejemplo, acumulación incorrecta de resultados o iteraciones duplicadas).

El impacto principal es la sobre-representación de ciertas observaciones, lo que puede sesgar el análisis operativo, especialmente en métricas como distribución de señales, confidence o trade_rate.

Para resolverlo en esta etapa, se eliminan los duplicados exactos del dataset de probabilidades mediante:

```python
df_proba = df_proba.drop_duplicates().copy()
```

Se recomienda adicionalmente revisar el pipeline de `random_forest` para identificar el origen de la duplicación y evitar que vuelva a ocurrir en futuras ejecuciones.


### **4. Resolución y comprobación**

In [40]:
df_proba = df_proba.drop_duplicates().copy()

In [41]:
dup_proba = df_proba.duplicated().sum()

print("Duplicados exactos en df_proba:", dup_proba)

Duplicados exactos en df_proba: 0


## **1.7. Comprobación de valores nulos (NaNs)**

En esta etapa se analiza la presencia de valores nulos en los datasets consolidados. Dado que los resultados provienen de múltiples modelos con configuraciones distintas, es esperable encontrar NaNs en columnas asociadas a hiperparámetros o metadata específica de cada modelo.

El objetivo no es eliminar estos valores, sino distinguir entre:

- NaNs esperables (columnas específicas de ciertos modelos)
- NaNs problemáticos (columnas clave para el análisis)

Se presta especial atención a que no existan valores nulos en variables críticas como métricas principales, probabilidades, etiquetas reales o variables operativas.

In [42]:
print('7. Validación de NaNs:\n')

print("df_metrics:")
display(df_metrics.isna().sum().sort_values(ascending=False))

print("\ndf_proba:")
display(df_proba.isna().sum().sort_values(ascending=False))

7. Validación de NaNs:

df_metrics:


,0
best_valid_loss,12
reg_alpha,12
subsample,12
max_features,12
min_samples_split,12
tree_method,12
gamma,12
min_child_weight,12
colsample_bytree,12
reg_lambda,12



df_proba:


,0
max_features,82584
min_samples_split,82584
min_samples_leaf,82584
best_epoch,78652
max_iter,78652
multi_class,78652
patience,78652
eval_batch_size,78652
C,78652
batch_size,78652


**Observaciones de la validación de NaNs**

En ambos datasets se observa una cantidad significativa de valores nulos en columnas asociadas a hiperparámetros. Esto es completamente esperable, ya que cada modelo utiliza configuraciones distintas y solo completa las columnas que le corresponden.

En `df_metrics`, los NaNs se concentran en parámetros específicos de cada tipo de modelo (por ejemplo, parámetros de redes neuronales, random forest o regresión logística). Las métricas principales, así como las columnas de identificación (`model`, `target`, `window_size`, `split`), no presentan valores nulos, lo cual confirma que el dataset es válido para análisis comparativo.

En `df_proba`, ocurre el mismo fenómeno a mayor escala debido al tamaño del dataset. Las columnas con mayor cantidad de NaNs corresponden a hiperparámetros y metadata que no aplican a todos los modelos. También se observan algunos NaNs en columnas como `split`, `window_size` e `input_mode`, lo cual es consistente con diferencias en los pipelines de generación de predicciones.

Las variables críticas para el análisis no presentan valores nulos, incluyendo:

* probabilidades (`proba_*`)
* predicción (`pred_label`)
* variable objetivo (`y_true`)
* confianza (`confidence`)
* variables operativas (`trade`, `signal_raw`, `is_correct`)

**Conclusión**

Los valores nulos observados son esperables y no afectan la validez del análisis. No es necesario realizar imputación ni limpieza adicional, siempre que se ignoren las columnas de hiperparámetros en los análisis globales.


## **1.8. Validación de baseline**

En esta etapa se verifica la consistencia del baseline utilizado para comparar el desempeño de los modelos. En particular, se analiza la métrica balanced_accuracy_naive, que representa el rendimiento de un clasificador base (por ejemplo, predicción constante o distribución uniforme).

El objetivo es asegurar que:

- el baseline esté correctamente calculado
- sea consistente entre modelos para un mismo target
- permita una comparación válida mediante las métricas de mejora (*_gain_vs_naive)

Dado que el baseline depende únicamente de la distribución del target, debería ser igual para todos los modelos que evalúan el mismo target.

In [43]:
print('8. Validación de baseline:\n')

print("Valores únicos globales:")
print(df_metrics["balanced_accuracy_naive"].unique())

print("\nValores por target:")
display(
    df_metrics.groupby("target")["balanced_accuracy_naive"].unique()
)

8. Validación de baseline:

Valores únicos globales:
[0.33333333]

Valores por target:


,balanced_accuracy_naive
target,
t2_p40_h30,[0.3333333333333333]
t2_p40_h60,[0.3333333333333333]
t2_p50_h30,[0.3333333333333333]


**Observaciones de la validación de baseline**

* El valor de `balanced_accuracy_naive` es único a nivel global (≈ 0.3333).
* El baseline es consistente entre todos los targets y modelos.
* El valor corresponde a un problema de clasificación ternaria balanceada (1/3).

**Conclusión**

El baseline está correctamente definido y es consistente, por lo que las métricas de mejora respecto al naive son válidas y comparables entre todos los modelos y targets.


## **1.9. Validación de consistencia de gain**


En esta etapa se verifica que la métrica de mejora respecto al baseline esté correctamente calculada. En particular, se comprueba que la diferencia entre el desempeño del modelo (`balanced_accuracy`) y el baseline (`balanced_accuracy_naive`) coincida con la columna de ganancia (`balanced_accuracy_gain_vs_naive`).

El objetivo es detectar posibles errores en el cálculo o en la construcción del dataset.


**Observación esperada**

* El valor debe ser 0 o muy cercano a 0





In [44]:
print('9. Validación de consistencia de gain:\n')

max_diff = (
    (df_metrics["balanced_accuracy"] - df_metrics["balanced_accuracy_naive"]
     - df_metrics["balanced_accuracy_gain_vs_naive"])
    .abs()
    .max()
)

print("Máxima diferencia:", max_diff)

9. Validación de consistencia de gain:

Máxima diferencia: 0.0


La diferencia es nula, se confirma que la métrica de ganancia está correctamente calculada y es consistente con las métricas base.

## **1.10. Validación final rápida**

En esta etapa se realiza una verificación general del dataset consolidado, con el objetivo de confirmar que la estructura final es coherente antes de avanzar al análisis.

Se revisa:

* el tamaño del dataset
* la cantidad de modelos distintos
* la cantidad de configuraciones únicas (modelo + window_size + target)

Esto permite detectar rápidamente inconsistencias globales en la construcción de los datos.

In [45]:
print('10. Validación final rápida:\n')

print("df_metrics:")
print("Shape:", df_metrics.shape)
print("Models:", df_metrics["model"].nunique())
print("Configs:", df_metrics.groupby(["model", "window_size", "target"]).ngroups)

print("\ndf_proba:")
print("Shape:", df_proba.shape)
print("Models:", df_proba["model"].nunique())
print("Configs:", df_proba.groupby(["model", "window_size", "target"]).ngroups)

10. Validación final rápida:

df_metrics:
Shape: (15, 58)
Models: 5
Configs: 15

df_proba:
Shape: (99298, 49)
Models: 5
Configs: 12


**Observaciones de la validación final**

* En `df_metrics`, la estructura es completamente consistente:

  * 15 filas → 5 modelos × 3 targets
  * 5 modelos detectados correctamente
  * 15 configuraciones únicas → esperado

* En `df_proba`, el tamaño del dataset es consistente luego de eliminar duplicados.

* Se mantienen los 5 modelos correctamente identificados.

* Se observan 12 configuraciones únicas en lugar de 15.

**Interpretación**

* La diferencia en configuraciones se debe a la eliminación de duplicados en `random_forest`.
* Al remover filas repetidas, algunas combinaciones quedaron con menor cantidad de registros, pero no desaparecieron targets completos.
* No es un problema estructural, sino una consecuencia directa de la limpieza aplicada.

**Conclusión**

El dataset consolidado es consistente y válido para continuar con el análisis. La diferencia en configuraciones en `df_proba` es esperable tras la eliminación de duplicados y no afecta la comparabilidad entre modelos.



# **2. Definición de métricas de decisión**

En esta etapa se establecen las métricas que guiarán la selección y comparación de modelos. El objetivo es definir un criterio claro, consistente y alineado con el problema, evitando conclusiones erróneas basadas en métricas aisladas o poco representativas.

Se adopta un enfoque combinado que considera tanto el desempeño predictivo como la utilidad práctica del modelo. En particular, se distingue entre:

* métricas de desempeño (qué tan bien predice el modelo)
* métricas relativas al baseline (si realmente aporta valor)
* métricas operativas (si el modelo es útil en un contexto de trading)

Este esquema permite evaluar los modelos no solo como clasificadores, sino como generadores de señales, lo cual es fundamental en un entorno de trading algorítmico.

En este contexto, la métrica principal de evaluación será el `balanced_accuracy`, complementada por la mejora respecto al baseline (`balanced_accuracy_gain_vs_naive`) como criterio de utilidad mínima. Adicionalmente, se incorporan métricas operativas derivadas de las probabilidades, que permiten evaluar el comportamiento real del modelo en términos de generación de señales.

La definición explícita de estas métricas constituye la base del proceso de selección y asegura que todas las comparaciones se realicen bajo un criterio objetivo y consistente.


## **2.1. Creación de flags de calidad**

En esta etapa se construyen indicadores simples que permitan clasificar rápidamente la calidad de cada configuración evaluada. El objetivo es transformar las métricas numéricas en señales de lectura más directa, facilitando el filtrado inicial de modelos antes del análisis más detallado.

Se distinguen dos tipos de flags:

* flags de desempeño, basados en la mejora respecto al baseline naive
* flags operativos, basados en la capacidad del modelo para generar señal utilizable

Los flags de desempeño permiten identificar si un modelo supera al baseline y si la magnitud de esa mejora es suficientemente relevante. Los flags operativos, por su parte, permiten detectar si el modelo genera señales con una frecuencia mínima aceptable y con una calidad razonable desde el punto de vista práctico.

Esta etapa no reemplaza el análisis cuantitativo posterior, pero ayuda a ordenar rápidamente el espacio de configuraciones y a descartar combinaciones débiles o poco operativas.






### Código de creación de flags

In [46]:
print("2.1. Creación de flags de calidad:\n")

df_flags = df_metrics.copy()

# =========================================
# Flags de desempeño
# =========================================
df_flags["is_useful"] = df_flags["balanced_accuracy_gain_vs_naive"] > 0
df_flags["is_strong"] = df_flags["balanced_accuracy_gain_vs_naive"] > 0.03
df_flags["is_very_strong"] = df_flags["balanced_accuracy_gain_vs_naive"] > 0.05

# =========================================
# Construcción de métricas operativas
# desde df_proba
# =========================================
df_oper = (
    df_proba
    .groupby(["model", "window_size", "target"], as_index=False)
    .agg(
        n_obs=("trade", "size"),
        trade_rate=("trade", "mean"),
        mean_confidence=("confidence", "mean"),
        mean_confidence_trade=("confidence", lambda x: x[df_proba.loc[x.index, "trade"]].mean()
                               if df_proba.loc[x.index, "trade"].any() else 0.0),
        precision_trade=("is_correct", lambda x: x[df_proba.loc[x.index, "trade"]].mean()
                         if df_proba.loc[x.index, "trade"].any() else 0.0),
        n_trades=("trade", "sum"),
    )
)

# =========================================
# Merge métricas + operatividad
# =========================================
df_flags = df_flags.merge(
    df_oper,
    on=["model", "window_size", "target"],
    how="left"
)

# =========================================
# Flags operativos
# =========================================
df_flags["has_signal"] = df_flags["trade_rate"] > 0.05
df_flags["is_precise"] = df_flags["precision_trade"] > 0.40
df_flags["is_operable"] = df_flags["has_signal"] & df_flags["is_precise"]

# =========================================
# Vista rápida
# =========================================
display(
    df_flags[
        [
            "model",
            "split",
            "window_size",
            "target",
            "balanced_accuracy",
            "balanced_accuracy_gain_vs_naive",
            "is_useful",
            "is_strong",
            "is_very_strong",
            "trade_rate",
            "precision_trade",
            "mean_confidence",
            "has_signal",
            "is_precise",
            "is_operable",
        ]
    ].sort_values(
        ["is_operable", "is_very_strong", "is_strong", "balanced_accuracy_gain_vs_naive"],
        ascending=False
    )
)

2.1. Creación de flags de calidad:



,model,split,window_size,target,balanced_accuracy,balanced_accuracy_gain_vs_naive,is_useful,is_strong,is_very_strong,trade_rate,precision_trade,mean_confidence,has_signal,is_precise,is_operable
0,gru,valid,30,t2_p40_h30,0.415750,0.082417,True,True,True,0.180035,0.455206,0.414847,True,True,True
14,xgboost,valid,30,t2_p50_h30,0.408889,0.075556,True,True,True,0.259808,0.406040,0.430655,True,True,True
12,xgboost,valid,30,t2_p40_h30,0.405681,0.072348,True,True,True,0.259954,0.434880,0.421458,True,True,True
5,logistic_regression,valid,30,t2_p50_h30,0.402972,0.069639,True,True,True,0.234525,0.420074,0.434151,True,True,True
2,gru,valid,30,t2_p50_h30,0.402873,0.069539,True,True,True,0.200668,0.432295,0.425129,True,True,True
3,logistic_regression,valid,30,t2_p40_h30,0.396396,0.063063,True,True,True,0.240046,0.448547,0.429001,True,True,True
6,random_forest,valid,30,t2_p40_h30,0.381023,0.047690,True,True,False,0.412965,0.406646,0.471550,True,True,True
8,random_forest,valid,30,t2_p50_h30,0.373990,0.040657,True,True,False,0.228080,0.404289,0.518256,True,True,True
1,gru,valid,30,t2_p40_h60,0.417276,0.083942,True,True,True,0.256466,0.379603,0.430914,True,False,False
10,transformer,valid,30,t2_p40_h60,0.410546,0.077213,True,True,True,NaN,NaN,NaN,False,False,False


Qué representa cada flag

* `is_useful`: el modelo supera al baseline naive
* `is_strong`: la mejora sobre el baseline ya es relevante
* `is_very_strong`: la señal es claramente fuerte
* `has_signal`: el modelo genera una frecuencia mínima de trades
* `is_precise`: las señales operativas muestran una precisión aceptable
* `is_operable`: el modelo combina frecuencia mínima y precisión razonable

---

Criterio adoptado en esta etapa

* modelo útil → `balanced_accuracy_gain_vs_naive > 0`
* modelo fuerte → `balanced_accuracy_gain_vs_naive > 0.03`
* modelo muy fuerte → `balanced_accuracy_gain_vs_naive > 0.05`
* modelo con señal → `trade_rate > 0.05`
* modelo operativo → señal suficiente + precisión aceptable

---

Observación

Los umbrales operativos (`trade_rate > 0.05`, `precision_trade > 0.40`) deben interpretarse como reglas iniciales de trabajo. No son definitivos, pero permiten construir un primer filtro razonable para identificar configuraciones prometedoras.

## **2.2. Revisión de la distribución de señal**

En esta etapa se analiza la distribución de la señal generada por los modelos, tanto desde el punto de vista predictivo como operativo. El objetivo es entender cuánta señal real existe en el dataset, cómo se distribuye entre las distintas configuraciones y si es suficientemente fuerte como para ser explotada.

Se evalúan dos dimensiones complementarias:

* señal predictiva: medida a través de la mejora respecto al baseline (`balanced_accuracy_gain_vs_naive`)
* señal operativa: medida a través de métricas derivadas de probabilidades (`trade_rate`, `precision_trade`, `confidence`)

Este análisis permite determinar si el problema contiene alpha real, si la señal está concentrada en pocas configuraciones o distribuida de forma homogénea, y si los modelos generan señales utilizables en la práctica.

---

**Interpretación esperada**

- **Señal predictiva**

  * Si la media del gain es mayor que 0 → existe señal real
  * Si la mediana es baja → la mayoría de configuraciones tienen señal débil
  * Si el percentil 75 es alto → hay un subconjunto con buena señal
  * Si el máximo es alto → existen configuraciones muy prometedoras

- **Señal operativa**

  * `trade_rate`: indica qué tan frecuentemente el modelo genera señales
  * `precision_trade`: mide la calidad de esas señales
  * `mean_confidence`: refleja la seguridad promedio del modelo

- **Distribución típica esperada**

  * muchas configuraciones con señal baja o nula
  * pocas configuraciones con señal fuerte
  * trade_rate bajo en promedio (modelos conservadores)
  * mayor valor en subconjuntos específicos

---

**Conclusión**

Este análisis permite entender la calidad global del espacio de modelos evaluado y proporciona el contexto necesario para interpretar correctamente los flags definidos en el punto anterior. A partir de aquí, es posible avanzar con filtrado de calidad y ranking con criterios mejor fundamentados.

### Código de análisis

In [48]:
print("2.2. Distribución de señal:\n")

# =========================================
# 1. Distribución de gain (predictiva)
# =========================================
print("Distribución de balanced_accuracy_gain_vs_naive:\n")

display(
    df_metrics["balanced_accuracy_gain_vs_naive"].describe()
)

# =========================================
# 2. Distribución por target
# =========================================
print("\nDistribución por target:\n")

display(
    df_metrics.groupby("target")["balanced_accuracy_gain_vs_naive"].describe()
)

# =========================================
# 3. Métricas operativas desde df_flags
# =========================================
print("\nDistribución de métricas operativas:\n")

display(
    df_flags[
        ["trade_rate", "precision_trade", "mean_confidence"]
    ].describe()
)

# =========================================
# 4. Distribución operativa por target
# =========================================
print("\nDistribución operativa por target:\n")

display(
    df_flags.groupby("target")[
        ["trade_rate", "precision_trade", "mean_confidence"]
    ].describe()
)

# =========================================
# 5. Conteo de flags
# =========================================
print("\nConteo de flags:\n")

display(
    df_flags[
        ["is_useful", "is_strong", "is_very_strong", "is_operable"]
    ].sum()
)

2.2. Distribución de señal:

Distribución de balanced_accuracy_gain_vs_naive:



,balanced_accuracy_gain_vs_naive
count,15.000000
mean,0.065457
std,0.015659
min,0.027650
25%,0.062514
50%,0.069639
75%,0.074916
max,0.083942



Distribución por target:



,count,mean,std,min,25%,50%,75%,max
target,,,,,,,,
t2_p40_h30,5.0,0.067216,0.012914,0.047690,0.063063,0.070563,0.072348,0.082417
t2_p40_h60,5.0,0.063221,0.021775,0.027650,0.061964,0.065336,0.077213,0.083942
t2_p50_h30,5.0,0.065933,0.014386,0.040657,0.069539,0.069639,0.074275,0.075556



Distribución de métricas operativas:



,trade_rate,precision_trade,mean_confidence
count,12.000000,12.000000,12.000000
mean,0.258237,0.407361,0.446793
std,0.057418,0.033157,0.033928
min,0.180035,0.351420,0.414847
25%,0.232914,0.387966,0.428033
50%,0.258137,0.406343,0.432532
75%,0.268075,0.432942,0.447886
max,0.412965,0.455206,0.518256



Distribución operativa por target:



trade_rate                                                   \
                count     mean       std       min       25%       50%   
target                                                                   
t2_p40_h30        4.0  0.27325  0.099144  0.180035  0.225044  0.250000   
t2_p40_h60        4.0  0.27069  0.012472  0.256466  0.262133  0.272128   
t2_p50_h30        4.0  0.23077  0.024294  0.200668  0.221227  0.231302   

                               precision_trade            ...            \
                 75%       max           count      mean  ...       75%   
target                                                    ...             
t2_p40_h30  0.298206  0.412965             4.0  0.436320  ...  0.450212   
t2_p40_h60  0.280685  0.282040             4.0  0.370089  ...  0.382391   
t2_p50_h30  0.240846  0.259808             4.0  0.415675  ...  0.423130   

                     mean_confidence                                          \
                 max           count      mean       std       min       25%   
target                                                                         
t2_p40_h30  0.455206             4.0  0.434214  0.025554  0.414847  0.419805   
t2_p40_h60  0.390754             4.0  0.454116  0.035868  0.430914  0.436193   
t2_p50_h30  0.432295             4.0  0.452048  0.044295  0.425129  0.429273   

                                          
                 50%       75%       max  
target                                    
t2_p40_h30  0.425229  0.439638  0.471550  
t2_p40_h60  0.438975  0.456899  0.507601  
t2_p50_h30  0.432403  0.455177  0.518256  

[3 rows x 24 columns]


Conteo de flags:



,0
is_useful,15
is_strong,14
is_very_strong,12
is_operable,8


### Observaciones de la distribución de señal

**Señal predictiva**

* Existe señal clara en el dataset: el gain medio es elevado (~0.065) y todos los modelos superan el baseline.
* La mediana (~0.069) es cercana al percentil 75, lo que indica que la mayoría de las configuraciones presentan buena señal.
* El valor mínimo (~0.027) es positivo, por lo que no existen configuraciones completamente inútiles.
* La dispersión es moderada, lo que sugiere estabilidad entre modelos.

Conclusión: el problema presenta una señal consistente y no se trata de un caso marginal.

---

**Análisis por target**

* `t2_p40_h30` y `t2_p50_h30` muestran señal estable y consistente.
* `t2_p40_h60` presenta mayor variabilidad, lo que indica mayor sensibilidad al modelo.
* Los valores máximos son similares entre targets, por lo que no hay un claro dominante en esta etapa.

Conclusión: los tres targets son válidos, aunque `h60` muestra menor estabilidad.

---

**Señal operativa**

* El `trade_rate` medio (~0.26) indica que los modelos generan señales con frecuencia razonable.
* La `precision_trade` (~0.40) es aceptable, aunque no elevada.
* La `confidence` (~0.44) es baja, lo que refleja que los modelos no son altamente seguros en sus predicciones.

Conclusión: existe señal operativa real, pero de intensidad moderada, lo que confirma que el problema es desafiante.

---

**Análisis operativo por target**

* `t2_p40_h30` presenta mayor dispersión en `trade_rate`, lo que indica diferencias relevantes entre modelos.
* `t2_p40_h60` es más estable, pero con menor precisión, lo que implica menor calidad de señal.
* `t2_p50_h30` muestra menor frecuencia de operación, pero mejor precisión.

**Conclusión:**

* `p50_h30` se comporta como un target más selectivo
* `p40_h30` es más activo
* `p40_h60` presenta menor calidad operativa

---

**Análisis de flags**

* `is_useful`: todos los modelos superan el baseline
* `is_strong`: la mayoría presenta señal relevante
* `is_very_strong`: gran parte muestra señal fuerte
* `is_operable`: aproximadamente la mitad de las configuraciones son utilizables operativamente

**Conclusión clave**

Existe una diferencia importante entre desempeño predictivo y utilidad operativa: muchos modelos presentan buen rendimiento en métricas, pero solo un subconjunto genera señales realmente utilizables.

---

**Conclusión general**

El dataset contiene señal real y consistente, y la mayoría de los modelos presentan buen desempeño predictivo. Sin embargo, solo una parte de ellos logra traducir esa capacidad en señales operativas útiles.

A partir de este punto, el foco del análisis debe desplazarse desde la evaluación puramente predictiva hacia la identificación de modelos que generen decisiones efectivas en un contexto operativo.


# **3. Análisis de probabilidades**


En esta etapa se analizan las salidas probabilísticas de los modelos con el objetivo de entender su comportamiento antes de aplicar cualquier filtro o criterio de selección. A diferencia de las métricas agregadas, este análisis permite observar cómo el modelo toma decisiones a nivel de observación individual.

Se evalúan los siguientes aspectos:

* distribución de `confidence`, para entender el nivel de seguridad de las predicciones
* distribución de clases predichas, para detectar posibles desbalances
* sesgo hacia la clase neutra (0), que puede indicar falta de señal
* calibración implícita de probabilidades, evaluando si valores altos de probabilidad corresponden efectivamente a predicciones correctas

Este análisis permite identificar problemas típicos como modelos que no generan señales, modelos colapsados a una única clase, baja diferenciación entre clases o sobreconfianza en las predicciones. Asimismo, proporciona información clave para interpretar correctamente las métricas operativas y definir reglas de decisión posteriores.


## **3.1. Código de análisis de probabilidades**

In [49]:
print("3. Análisis de probabilidades:\n")

# =========================================
# 3.1 Distribución global de confidence
# =========================================
print("3.1. Distribución global de confidence:\n")
display(df_proba["confidence"].describe())

print("\nConfidence por modelo:")
display(df_proba.groupby("model")["confidence"].describe())

print("\nConfidence por target:")
display(df_proba.groupby("target")["confidence"].describe())


# =========================================
# 3.2 Distribución de clases predichas
# =========================================
print("\n3.2. Distribución de clases predichas:\n")

pred_dist_model = (
    df_proba.groupby("model")["pred_label"]
    .value_counts(normalize=True)
    .rename("pct")
    .reset_index()
    .sort_values(["model", "pred_label"])
)

pred_dist_target = (
    df_proba.groupby("target")["pred_label"]
    .value_counts(normalize=True)
    .rename("pct")
    .reset_index()
    .sort_values(["target", "pred_label"])
)

print("Distribución de clases predichas por modelo:")
display(pred_dist_model)

print("\nDistribución de clases predichas por target:")
display(pred_dist_target)


# =========================================
# 3.3 Sesgo hacia clase neutra (0)
# =========================================
print("\n3.3. Sesgo hacia clase neutra (0):\n")

neutral_bias_model = (
    df_proba.assign(is_neutral_pred=df_proba["pred_label"] == 0)
    .groupby("model", as_index=False)
    .agg(
        neutral_pred_rate=("is_neutral_pred", "mean"),
        trade_rate=("trade", "mean"),
        mean_confidence=("confidence", "mean"),
    )
    .sort_values("neutral_pred_rate", ascending=False)
)

neutral_bias_target = (
    df_proba.assign(is_neutral_pred=df_proba["pred_label"] == 0)
    .groupby("target", as_index=False)
    .agg(
        neutral_pred_rate=("is_neutral_pred", "mean"),
        trade_rate=("trade", "mean"),
        mean_confidence=("confidence", "mean"),
    )
    .sort_values("neutral_pred_rate", ascending=False)
)

print("Sesgo neutral por modelo:")
display(neutral_bias_model)

print("\nSesgo neutral por target:")
display(neutral_bias_target)


# =========================================
# 3.4 Calibración implícita de probabilidades
# =========================================
print("\n3.4. Calibración implícita de probabilidades:\n")

# bins de confidence
df_calib = df_proba.copy()
df_calib["confidence_bin"] = pd.cut(
    df_calib["confidence"],
    bins=[0.0, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.70, 1.00],
    include_lowest=True
)

calibration_global = (
    df_calib.groupby("confidence_bin", observed=False)
    .agg(
        n_obs=("is_correct", "size"),
        acc_real=("is_correct", "mean"),
        confidence_mean=("confidence", "mean"),
        trade_rate=("trade", "mean"),
    )
    .reset_index()
)

calibration_model = (
    df_calib.groupby(["model", "confidence_bin"], observed=False)
    .agg(
        n_obs=("is_correct", "size"),
        acc_real=("is_correct", "mean"),
        confidence_mean=("confidence", "mean"),
    )
    .reset_index()
)

print("Calibración global por bin de confidence:")
display(calibration_global)

print("\nCalibración por modelo y bin de confidence:")
display(calibration_model)


# =========================================
# 3.5 Resumen rápido por modelo
# =========================================
print("\n3.5. Resumen rápido por modelo:\n")

proba_summary_model = (
    df_proba.assign(
        is_neutral_pred=df_proba["pred_label"] == 0,
        is_long_pred=df_proba["pred_label"] == 1,
        is_short_pred=df_proba["pred_label"] == -1,
    )
    .groupby("model", as_index=False)
    .agg(
        n_obs=("pred_label", "size"),
        mean_confidence=("confidence", "mean"),
        median_confidence=("confidence", "median"),
        pct_pred_short=("is_short_pred", "mean"),
        pct_pred_neutral=("is_neutral_pred", "mean"),
        pct_pred_long=("is_long_pred", "mean"),
        trade_rate=("trade", "mean"),
        accuracy_real=("is_correct", "mean"),
    )
    .sort_values("mean_confidence", ascending=False)
)

display(proba_summary_model)

3. Análisis de probabilidades:

3.1. Distribución global de confidence:



,confidence
count,99298.000000
mean,0.442418
std,0.078153
min,0.333675
25%,0.385765
50%,0.423544
75%,0.480000
max,0.997280



Confidence por modelo:


,count,mean,std,min,25%,50%,75%,max
model,,,,,,,,
gru,20646.0,0.423630,0.059186,0.333684,0.378223,0.410696,0.458215,0.724068
logistic_regression,20646.0,0.434383,0.065232,0.334468,0.386215,0.421581,0.470807,0.997280
random_forest,16714.0,0.499175,0.105668,0.335000,0.420000,0.475000,0.560000,0.960000
transformer,20646.0,0.435688,0.067298,0.334468,0.387110,0.420775,0.468012,0.984046
xgboost,20646.0,0.430022,0.069475,0.333675,0.378444,0.412067,0.463682,0.760036



Confidence por target:


...

,confidence_bin,n_obs,acc_real,confidence_mean,trade_rate
0,"(-0.001, 0.35]",3286,0.349361,0.344406,0.000000
1,"(0.35, 0.4]",31442,0.383182,0.376886,0.006170
2,"(0.4, 0.45]",28757,0.390409,0.423491,0.561255
3,"(0.45, 0.5]",17130,0.407531,0.473485,0.374606
4,"(0.5, 0.55]",9574,0.420305,0.522453,0.293190
5,"(0.55, 0.6]",4697,0.457313,0.573304,0.236534
6,"(0.6, 0.7]",3300,0.467273,0.639949,0.179394
7,"(0.7, 1.0]",1112,0.459532,0.767725,0.172662



Calibración por modelo y bin de confidence:


,model,confidence_bin,n_obs,acc_real,confidence_mean
0,gru,"(-0.001, 0.35]",937,0.375667,0.344037
1,gru,"(0.35, 0.4]",7829,0.388811,0.375879
2,gru,"(0.4, 0.45]",6014,0.394413,0.422622
3,gru,"(0.45, 0.5]",3425,0.428029,0.473336
4,gru,"(0.5, 0.55]",1711,0.459380,0.519778
5,gru,"(0.55, 0.6]",533,0.549719,0.571985
6,gru,"(0.6, 0.7]",183,0.672131,0.623652
7,gru,"(0.7, 1.0]",14,1.000000,0.713232
8,logistic_regression,"(-0.001, 0.35]",586,0.334471,0.344307
9,logistic_regression,"(0.35, 0.4]",6692,0.374626,0.377221



3.5. Resumen rápido por modelo:



,model,n_obs,mean_confidence,median_confidence,pct_pred_short,pct_pred_neutral,pct_pred_long,trade_rate,accuracy_real
2,random_forest,16714,0.499175,0.475000,0.164353,0.625165,0.210482,0.306988,0.376212
3,transformer,20646,0.435688,0.420775,0.311731,0.402935,0.285334,0.355323,0.405696
1,logistic_regression,20646,0.434383,0.421581,0.222125,0.553328,0.224547,0.246198,0.397753
4,xgboost,20646,0.430022,0.412067,0.306113,0.431173,0.262714,0.267267,0.402160
0,gru,20646,0.423630,0.410696,0.260680,0.529303,0.210016,0.212390,0.409280


## **3.2. Observaciones del análisis de probabilidades**

**Distribución de confidence**

* La confidence media (~0.44) es baja, con mediana ~0.42, lo que indica que los modelos no son altamente seguros en sus predicciones.
* La mayoría de los valores se concentran entre 0.35 y 0.48, reflejando probabilidades cercanas a un escenario casi aleatorio.
* Existen valores extremos cercanos a 1, pero son poco frecuentes, por lo que no representan el comportamiento típico.

- **Por modelo**

  * `random_forest` presenta la mayor confidence media (~0.50), significativamente superior al resto.
  * El resto de los modelos se agrupa en un rango estrecho (~0.42–0.44), mostrando comportamientos muy similares.
  * Esto sugiere que `random_forest` es más “seguro”, aunque esto no implica necesariamente mayor precisión.

---

**Distribución de clases predichas**

* Existe un sesgo general hacia la clase neutra (0), que concentra entre ~40% y ~62% de las predicciones según el modelo.
* `random_forest` es el más sesgado hacia la clase neutra (~62%).
* `transformer` y `xgboost` presentan distribuciones más balanceadas entre clases.

- **Por target**

  * La clase neutra domina en todos los targets (~47%–53%).
  * No se observa un colapso total a una sola clase, lo cual es positivo.

**Conclusión**

- Existe sesgo hacia la neutralidad, pero no es extremo en todos los modelos.

---

**Sesgo hacia clase neutra**

* `random_forest` presenta el mayor sesgo neutral (~0.62), acompañado de un trade_rate relativamente alto (~0.30).
* `transformer` es el menos sesgado (~0.40), y a su vez el que más opera (~0.35).
* `gru` y `logistic_regression` muestran comportamiento intermedio.

**Conclusión**

  Existe una relación clara:

  * menor sesgo neutro → mayor actividad (trade_rate)
  * mayor sesgo neutro → modelo más conservador

---

Calibración de probabilidades

* A nivel global, la accuracy real aumenta con la confidence, pero de forma débil.
* Incluso en rangos altos de confidence (>0.7), la accuracy no supera ~0.46–0.47 en promedio.
* Esto indica que los modelos tienden a ser **sobreconfiados**.

Por modelo

* `gru` muestra mejor comportamiento en rangos altos (mejor calibración en confidence alta).
* `random_forest` es claramente sobreconfiado: alta confidence pero baja accuracy relativa.
* `transformer` y `xgboost` presentan comportamiento intermedio.

Conclusión

La calibración es imperfecta en todos los modelos, con tendencia general a sobreestimar la probabilidad.

---

Resumen por modelo

* `random_forest`:

  * mayor confidence
  * mayor sesgo neutro
  * menor accuracy (~0.376)
    → modelo sobreconfiado y conservador

* `transformer`:

  * mejor balance de clases
  * mayor trade_rate
  * buena accuracy (~0.405)
    → modelo más activo y equilibrado

* `xgboost`:

  * balanceado
  * buen trade_rate
  * accuracy sólida (~0.402)
    → modelo robusto

* `gru`:

  * menor confidence
  * menor trade_rate
  * buena accuracy (~0.409)
    → modelo conservador pero consistente

* `logistic_regression`:

  * comportamiento intermedio
  * menor performance relativa

---

Conclusión general

* Los modelos muestran baja confianza en general, lo que refleja la dificultad del problema.
* Existe sesgo hacia la clase neutra, aunque no extremo en todos los casos.
* La calibración es imperfecta y tiende a la sobreconfianza, especialmente en `random_forest`.
* Los modelos más interesantes desde el punto de vista operativo son aquellos con menor sesgo neutro y mayor trade_rate (`transformer` y `xgboost`).

El análisis confirma que no basta con evaluar métricas agregadas: el comportamiento probabilístico introduce diferencias importantes en cómo cada modelo genera señales.


# **4. Filtro de calidad (anti-ruido)**


En esta etapa se filtran las configuraciones con el objetivo de eliminar ruido y conservar únicamente aquellas que presentan señal real y utilidad operativa.

Se adopta un enfoque combinado que integra criterios de desempeño predictivo y comportamiento operativo. A diferencia del enfoque original, la validación se realiza sobre el conjunto `valid`, dado que no se dispone de evaluación en `test` en esta etapa.

Se aplican los siguientes criterios:

a) Selección inicial

* Trabajar sobre `split = valid`
* Aplicar un umbral mínimo de señal:

  * `balanced_accuracy_gain_vs_naive > 0.01`

Este filtro elimina configuraciones con señal marginal o ruido puro.

---

b) Identificación de señal relevante

Se clasifican las configuraciones según la magnitud de la señal:

* señal moderada:

  * `gain > 0.03`
* señal fuerte:

  * `gain > 0.05`

Esto permite priorizar configuraciones con mayor potencial predictivo.

---

c) Filtro operativo

Se incorporan criterios basados en el comportamiento real del modelo:

* frecuencia mínima de señal:

  * `trade_rate > 0.05`
* calidad mínima de señal:

  * `precision_trade > 0.40`

Estos criterios aseguran que el modelo no solo predice bien, sino que genera señales utilizables.

---

d) Eliminación de configuraciones no operativas

Se descartan configuraciones que:

* no superan el baseline (`gain <= 0`)
* presentan muy baja frecuencia de operación
* presentan baja precisión en las señales

Este paso elimina modelos que, aunque correctos en métricas agregadas, no son útiles en la práctica.

---

e) Resultado esperado

Se obtiene un subconjunto de configuraciones que cumplen simultáneamente:

* señal predictiva real
* magnitud de señal relevante
* comportamiento operativo aceptable

Este dataset filtrado constituye la base para el ranking y la selección final de modelos.

---

**Conclusión**

Este filtro permite pasar de un conjunto amplio de configuraciones a un subconjunto más reducido, compuesto por modelos con señal consistente y potencial operativo. A partir de este punto, el análisis se enfoca en comparar y rankear únicamente configuraciones de calidad.


## **4.1. Código de filtro de calidad**

In [50]:
# ================================
# 4. Filtro de calidad (anti-ruido)
# ================================

df_filt = df_flags.copy()

# =========================================
# a) Selección inicial (solo valid)
# =========================================
df_filtered = df_filt[df_filt["split"] == "valid"].copy()

# =========================================
# b) Filtro mínimo de señal (anti-ruido)
# =========================================
df_filtered = df_filtered[
    df_filtered["balanced_accuracy_gain_vs_naive"] > 0.01
].copy()

# =========================================
# c) Flags de magnitud de señal
# =========================================
df_filtered["is_strong"] = (
    df_filtered["balanced_accuracy_gain_vs_naive"] > 0.03
)

df_filtered["is_very_strong"] = (
    df_filtered["balanced_accuracy_gain_vs_naive"] > 0.05
)

# =========================================
# d) Filtro operativo
# =========================================
df_filtered["has_signal"] = df_filtered["trade_rate"] > 0.05
df_filtered["is_precise"] = df_filtered["precision_trade"] > 0.40

df_filtered["is_operable"] = (
    df_filtered["has_signal"] & df_filtered["is_precise"]
)

# =========================================
# e) Filtro final (opcional, podés comentar si querés ver todo)
# =========================================
df_filtered = df_filtered[
    df_filtered["is_operable"]
].copy()

# =========================================
# f) Ordenar por calidad de señal
# =========================================
df_filtered = df_filtered.sort_values(
    by=[
        "is_very_strong",
        "balanced_accuracy_gain_vs_naive",
        "precision_trade",
        "trade_rate"
    ],
    ascending=False
).reset_index(drop=True)

# =========================================
# g) Resultados
# =========================================
print("Shape original:", df_flags.shape)
print("Shape filtrado:", df_filtered.shape)

display(
    df_filtered[
        [
            "model",
            "window_size",
            "target",
            "balanced_accuracy",
            "balanced_accuracy_gain_vs_naive",
            "trade_rate",
            "precision_trade",
            "mean_confidence",
            "is_strong",
            "is_very_strong",
            "is_operable",
        ]
    ]
)

Shape original: (15, 70)
Shape filtrado: (8, 70)


,model,window_size,target,balanced_accuracy,balanced_accuracy_gain_vs_naive,trade_rate,precision_trade,mean_confidence,is_strong,is_very_strong,is_operable
0,gru,30,t2_p40_h30,0.415750,0.082417,0.180035,0.455206,0.414847,True,True,True
1,xgboost,30,t2_p50_h30,0.408889,0.075556,0.259808,0.406040,0.430655,True,True,True
2,xgboost,30,t2_p40_h30,0.405681,0.072348,0.259954,0.434880,0.421458,True,True,True
3,logistic_regression,30,t2_p50_h30,0.402972,0.069639,0.234525,0.420074,0.434151,True,True,True
4,gru,30,t2_p50_h30,0.402873,0.069539,0.200668,0.432295,0.425129,True,True,True
5,logistic_regression,30,t2_p40_h30,0.396396,0.063063,0.240046,0.448547,0.429001,True,True,True
6,random_forest,30,t2_p40_h30,0.381023,0.047690,0.412965,0.406646,0.471550,True,False,True
7,random_forest,30,t2_p50_h30,0.373990,0.040657,0.228080,0.404289,0.518256,True,False,True


## **4.2. Observaciones de filtrado**

* El filtro reduce el conjunto de 15 a 8 configuraciones, eliminando aproximadamente la mitad de los modelos.
* Todas las configuraciones restantes presentan señal fuerte (`is_strong = True`) y la mayoría también señal muy fuerte.
* Se mantienen múltiples modelos, lo que indica que no hay un único dominante en esta etapa.

Por modelo

* `gru`, `xgboost` y `logistic_regression` aparecen varias veces → comportamiento consistente entre targets.
* `random_forest` permanece, pero solo en configuraciones con menor fuerza de señal (no `is_very_strong`).
* `transformer` queda completamente eliminado → no cumple criterios operativos.

Por target

* `t2_p40_h30` y `t2_p50_h30` dominan el conjunto filtrado.
* `t2_p40_h60` desaparece completamente → indica menor calidad operativa.

Métricas operativas

* `trade_rate` en rango razonable (~0.18 a ~0.41), sin modelos “apagados”.
* `precision_trade` consistente (~0.40–0.45), alineada con lo observado previamente.
* `confidence` moderada, sin valores extremos → coherente con análisis anterior.

Conclusión

El filtro elimina configuraciones débiles y deja un subconjunto consistente, con señal fuerte y comportamiento operativo aceptable. A partir de este punto, el análisis puede enfocarse en comparar estos candidatos para identificar las mejores combinaciones.



# **5. Ranking por modelo (nivel arquitectura)**


En esta etapa se evalúa el desempeño de cada arquitectura de modelo de forma agregada, utilizando únicamente las configuraciones que superaron el filtro de calidad. El objetivo es obtener una visión global de qué tipos de modelos capturan mejor la señal del problema, considerando tanto su desempeño predictivo como su comportamiento operativo.

Al agrupar por modelo y promediar sus métricas, se logra una comparación más robusta entre arquitecturas, reduciendo el efecto de variaciones puntuales entre targets. Esto permite identificar qué modelos presentan mejor equilibrio entre calidad de predicción, consistencia y utilidad práctica.

Se analizan principalmente:

* promedio de métricas de desempeño (`balanced_accuracy`, `balanced_accuracy_gain_vs_naive`, `f1_macro`)
* métricas operativas promedio (`trade_rate`, `precision_trade`, `confidence`)
* estabilidad entre targets (variabilidad de métricas dentro de cada modelo)

Este análisis permite:

* comparar arquitecturas de forma justa
* identificar modelos consistentemente buenos
* descartar arquitecturas inestables o débiles
* seleccionar los modelos más prometedores para el siguiente paso del análisis y eventual tuning

En esta etapa no se elige una configuración específica, sino que se evalúa el comportamiento general de cada tipo de modelo.


## **5.1. Código para ranking de modelos por arquitectura**

In [52]:
# =========================================
# 5. Ranking por modelo (nivel arquitectura)
# =========================================

print("5. Ranking por modelo (nivel arquitectura):\n")

# Se trabaja sobre configuraciones ya filtradas
df_rank_model = df_filtered.copy()

# =========================================
# 1. Ranking agregado por modelo
# =========================================
ranking_model = (
    df_rank_model
    .groupby("model", as_index=False)
    .agg(
        n_configs=("target", "count"),
        n_targets=("target", "nunique"),

        balanced_accuracy_mean=("balanced_accuracy", "mean"),
        balanced_accuracy_std=("balanced_accuracy", "std"),

        gain_mean=("balanced_accuracy_gain_vs_naive", "mean"),
        gain_std=("balanced_accuracy_gain_vs_naive", "std"),

        f1_macro_mean=("f1_macro", "mean"),
        f1_macro_std=("f1_macro", "std"),

        trade_rate_mean=("trade_rate", "mean"),
        trade_rate_std=("trade_rate", "std"),

        precision_trade_mean=("precision_trade", "mean"),
        precision_trade_std=("precision_trade", "std"),

        confidence_mean=("mean_confidence", "mean"),
        confidence_std=("mean_confidence", "std"),
    )
    .sort_values(
        by=["gain_mean", "precision_trade_mean", "trade_rate_mean"],
        ascending=False
    )
    .reset_index(drop=True)
)

print("Ranking agregado por modelo:")
display(ranking_model)

# =========================================
# 2. Detalle por modelo y target
# =========================================
detail_model_target = (
    df_rank_model[
        [
            "model",
            "target",
            "balanced_accuracy",
            "balanced_accuracy_gain_vs_naive",
            "f1_macro",
            "trade_rate",
            "precision_trade",
            "mean_confidence",
        ]
    ]
    .sort_values(["model", "balanced_accuracy_gain_vs_naive"], ascending=[True, False])
    .reset_index(drop=True)
)

print("\nDetalle por modelo y target:")
display(detail_model_target)

# =========================================
# 3. Ranking simple por gain promedio
# =========================================
print("\nRanking simple por gain promedio:")
display(
    ranking_model[
        [
            "model",
            "n_configs",
            "n_targets",
            "gain_mean",
            "balanced_accuracy_mean",
            "f1_macro_mean",
            "trade_rate_mean",
            "precision_trade_mean",
            "confidence_mean",
        ]
    ].sort_values("gain_mean", ascending=False).reset_index(drop=True)
)

# =========================================
# 4. Chequeo de estabilidad
# =========================================
print("\nChequeo de estabilidad por modelo:")
display(
    ranking_model[
        [
            "model",
            "gain_mean",
            "gain_std",
            "balanced_accuracy_mean",
            "balanced_accuracy_std",
            "trade_rate_mean",
            "trade_rate_std",
            "precision_trade_mean",
            "precision_trade_std",
        ]
    ].sort_values(["gain_std", "precision_trade_std"], ascending=True).reset_index(drop=True)
)

5. Ranking por modelo (nivel arquitectura):

Ranking agregado por modelo:


,model,n_configs,n_targets,balanced_accuracy_mean,balanced_accuracy_std,gain_mean,gain_std,f1_macro_mean,f1_macro_std,trade_rate_mean,trade_rate_std,precision_trade_mean,precision_trade_std,confidence_mean,confidence_std
0,gru,2,2,0.409311,0.009106,0.075978,0.009106,0.383624,0.002544,0.190352,0.014590,0.443751,0.016200,0.419988,0.007270
1,xgboost,2,2,0.407285,0.002268,0.073952,0.002268,0.394488,0.013138,0.259881,0.000103,0.420460,0.020393,0.426056,0.006503
2,logistic_regression,2,2,0.399684,0.004650,0.066351,0.004650,0.377584,0.015199,0.237286,0.003904,0.434311,0.020133,0.431576,0.003642
3,random_forest,2,2,0.377507,0.004973,0.044173,0.004973,0.340696,0.034809,0.320522,0.130734,0.405468,0.001667,0.494903,0.033026



Detalle por modelo y target:


,model,target,balanced_accuracy,balanced_accuracy_gain_vs_naive,f1_macro,trade_rate,precision_trade,mean_confidence
0,gru,t2_p40_h30,0.415750,0.082417,0.381825,0.180035,0.455206,0.414847
1,gru,t2_p50_h30,0.402873,0.069539,0.385423,0.200668,0.432295,0.425129
2,logistic_regression,t2_p50_h30,0.402972,0.069639,0.388331,0.234525,0.420074,0.434151
3,logistic_regression,t2_p40_h30,0.396396,0.063063,0.366836,0.240046,0.448547,0.429001
4,random_forest,t2_p40_h30,0.381023,0.047690,0.365310,0.412965,0.406646,0.471550
5,random_forest,t2_p50_h30,0.373990,0.040657,0.316082,0.228080,0.404289,0.518256
6,xgboost,t2_p50_h30,0.408889,0.075556,0.403778,0.259808,0.406040,0.430655
7,xgboost,t2_p40_h30,0.405681,0.072348,0.385198,0.259954,0.434880,0.421458



Ranking simple por gain promedio:


,model,n_configs,n_targets,gain_mean,balanced_accuracy_mean,f1_macro_mean,trade_rate_mean,precision_trade_mean,confidence_mean
0,gru,2,2,0.075978,0.409311,0.383624,0.190352,0.443751,0.419988
1,xgboost,2,2,0.073952,0.407285,0.394488,0.259881,0.420460,0.426056
2,logistic_regression,2,2,0.066351,0.399684,0.377584,0.237286,0.434311,0.431576
3,random_forest,2,2,0.044173,0.377507,0.340696,0.320522,0.405468,0.494903



Chequeo de estabilidad por modelo:


,model,gain_mean,gain_std,balanced_accuracy_mean,balanced_accuracy_std,trade_rate_mean,trade_rate_std,precision_trade_mean,precision_trade_std
0,xgboost,0.073952,0.002268,0.407285,0.002268,0.259881,0.000103,0.420460,0.020393
1,logistic_regression,0.066351,0.004650,0.399684,0.004650,0.237286,0.003904,0.434311,0.020133
2,random_forest,0.044173,0.004973,0.377507,0.004973,0.320522,0.130734,0.405468,0.001667
3,gru,0.075978,0.009106,0.409311,0.009106,0.190352,0.014590,0.443751,0.016200


## **5.2. Observaciones del ranking por modelo**

Desempeño predictivo

* `gru` presenta el mayor gain promedio (~0.076), seguido muy de cerca por `xgboost` (~0.074).
* `xgboost` muestra un desempeño muy competitivo, con métricas prácticamente equivalentes a `gru`.
* `logistic_regression` se mantiene como una alternativa sólida, aunque ligeramente inferior en gain.
* `random_forest` queda claramente por debajo en todas las métricas de desempeño.

Conclusión: `gru` y `xgboost` lideran en capacidad predictiva.

---

Métricas operativas

* `xgboost` presenta un buen equilibrio entre frecuencia (`trade_rate ~0.26`) y precisión (~0.42).
* `gru` muestra la mayor precisión (~0.44), pero con menor frecuencia (~0.19), lo que lo hace más conservador.
* `logistic_regression` mantiene un balance intermedio entre frecuencia y precisión.
* `random_forest` presenta la mayor frecuencia (~0.32), pero con menor precisión, lo que indica señales más ruidosas.

Conclusión:

* `gru` es más preciso pero menos activo
* `xgboost` es más equilibrado
* `random_forest` prioriza cantidad sobre calidad

---

Estabilidad

* `xgboost` es el modelo más estable (menor desviación en gain y métricas).
* `logistic_regression` también muestra buena estabilidad.
* `gru` presenta mayor variabilidad entre targets, aunque con mejor desempeño promedio.
* `random_forest` es inestable en `trade_rate`, con alta dispersión.

Conclusión: `xgboost` es el modelo más consistente.

---

Detalle por target

* Todos los modelos rinden mejor en `t2_p40_h30` y `t2_p50_h30`, confirmando lo observado previamente.
* `gru` y `xgboost` mantienen buen desempeño en ambos targets, lo que indica robustez.
* `random_forest` es consistentemente inferior en ambos casos.

---

Conclusión general

* `gru` y `xgboost` son los modelos más fuertes:

  * `gru` lidera en calidad de señal
  * `xgboost` destaca por estabilidad y equilibrio operativo

* `logistic_regression` es una alternativa competitiva, especialmente por su estabilidad.

* `random_forest` queda relegado debido a menor calidad de señal y mayor ruido operativo.

El análisis sugiere que los candidatos más prometedores para la siguiente etapa son `gru` y `xgboost`, con `logistic_regression` como referencia estable.


# **6. Análisis por window_size (L)**


En esta etapa se evalúa el impacto del tamaño de la ventana temporal (`window_size`) en el desempeño de los modelos, con el objetivo de identificar qué escala temporal captura mejor la señal del mercado.

Sin embargo, en el presente experimento se ha trabajado con un único valor de ventana (`L = 30`) para todos los modelos, con el fin de mantener condiciones controladas y comparables entre arquitecturas.

Como consecuencia, no es posible realizar un análisis comparativo sobre el efecto del tamaño de ventana en esta etapa, ni evaluar posibles efectos de sobreajuste temporal o pérdida de señal asociados a distintas escalas.

Este análisis queda reservado para futuras iteraciones del proceso, donde se incorporen múltiples valores de `window_size` que permitan estudiar de forma explícita su impacto en la calidad predictiva y operativa de los modelos.


# **7. Análisis por target**

En esta etapa se evalúa el desempeño de los distintos targets definidos, con el objetivo de identificar cuál de ellos captura mejor la señal del mercado y resulta más adecuado para la construcción de modelos predictivos y operativos.

A diferencia del enfoque inicial, donde el análisis se centraba únicamente en la predictibilidad según el horizonte, en esta etapa se adopta una visión más completa que incorpora tanto desempeño predictivo como comportamiento operativo.

Se analizan los siguientes aspectos para cada target:

* calidad predictiva, medida a través de métricas como `balanced_accuracy` y `balanced_accuracy_gain_vs_naive`
* comportamiento operativo, evaluado mediante `trade_rate`, `precision_trade` y `confidence`
* estabilidad de resultados entre distintos modelos

Este análisis permite determinar:

* qué horizonte y nivel de exigencia (percentil) generan mayor separabilidad de clases
* si la señal es consistente entre modelos o depende fuertemente de la arquitectura
* si el target produce señales utilizables en la práctica o solo mejoras marginales en métricas

El objetivo final es identificar los targets que combinan mejor capacidad predictiva, estabilidad y utilidad operativa, estableciendo así una base sólida para la selección de modelos en las siguientes etapas.



## **7.1. Código para análisis por target**

In [54]:
# =========================================
# 7. Análisis por target (TODOS los targets)
# =========================================

print("7. Análisis por target (completo):\n")

df_rank_target = df_flags.copy()

# =========================================
# 1. Ranking agregado por target
# =========================================
ranking_target = (
    df_rank_target
    .groupby("target", as_index=False)
    .agg(
        n_configs=("model", "count"),
        n_models=("model", "nunique"),

        balanced_accuracy_mean=("balanced_accuracy", "mean"),
        balanced_accuracy_std=("balanced_accuracy", "std"),

        gain_mean=("balanced_accuracy_gain_vs_naive", "mean"),
        gain_std=("balanced_accuracy_gain_vs_naive", "std"),

        f1_macro_mean=("f1_macro", "mean"),
        f1_macro_std=("f1_macro", "std"),

        trade_rate_mean=("trade_rate", "mean"),
        trade_rate_std=("trade_rate", "std"),

        precision_trade_mean=("precision_trade", "mean"),
        precision_trade_std=("precision_trade", "std"),

        confidence_mean=("mean_confidence", "mean"),
        confidence_std=("mean_confidence", "std"),
    )
    .sort_values(
        by=["gain_mean", "precision_trade_mean", "trade_rate_mean"],
        ascending=False
    )
    .reset_index(drop=True)
)

print("Ranking agregado por target:")
display(ranking_target)

# =========================================
# 2. Detalle por target y modelo
# =========================================
detail_target_model = (
    df_rank_target[
        [
            "target",
            "model",
            "balanced_accuracy",
            "balanced_accuracy_gain_vs_naive",
            "trade_rate",
            "precision_trade",
            "mean_confidence",
            "is_operable",
        ]
    ]
    .sort_values(["target", "balanced_accuracy_gain_vs_naive"], ascending=[True, False])
    .reset_index(drop=True)
)

print("\nDetalle por target y modelo:")
display(detail_target_model)

# =========================================
# 3. Ranking simple
# =========================================
print("\nRanking simple por target:")
display(
    ranking_target[
        [
            "target",
            "n_configs",
            "n_models",
            "gain_mean",
            "balanced_accuracy_mean",
            "trade_rate_mean",
            "precision_trade_mean",
            "confidence_mean",
        ]
    ].sort_values("gain_mean", ascending=False)
)

# =========================================
# 4. Estabilidad
# =========================================
print("\nEstabilidad por target:")
display(
    ranking_target[
        [
            "target",
            "gain_mean",
            "gain_std",
            "trade_rate_std",
            "precision_trade_std",
        ]
    ].sort_values("gain_std")
)

7. Análisis por target (completo):

Ranking agregado por target:


,target,n_configs,n_models,balanced_accuracy_mean,balanced_accuracy_std,gain_mean,gain_std,f1_macro_mean,f1_macro_std,trade_rate_mean,trade_rate_std,precision_trade_mean,precision_trade_std,confidence_mean,confidence_std
0,t2_p40_h30,5,5,0.400549,0.012914,0.067216,0.012914,0.377428,0.010605,0.27325,0.099144,0.436320,0.021516,0.434214,0.025554
1,t2_p50_h30,5,5,0.399267,0.014386,0.065933,0.014386,0.380074,0.036965,0.23077,0.024294,0.415675,0.013141,0.452048,0.044295
2,t2_p40_h60,5,5,0.396555,0.021775,0.063221,0.021775,0.383801,0.035342,0.27069,0.012472,0.370089,0.018245,0.454116,0.035868



Detalle por target y modelo:


,target,model,balanced_accuracy,balanced_accuracy_gain_vs_naive,trade_rate,precision_trade,mean_confidence,is_operable
0,t2_p40_h30,gru,0.415750,0.082417,0.180035,0.455206,0.414847,True
1,t2_p40_h30,xgboost,0.405681,0.072348,0.259954,0.434880,0.421458,True
2,t2_p40_h30,transformer,0.403897,0.070563,NaN,NaN,NaN,False
3,t2_p40_h30,logistic_regression,0.396396,0.063063,0.240046,0.448547,0.429001,True
4,t2_p40_h30,random_forest,0.381023,0.047690,0.412965,0.406646,0.471550,True
5,t2_p40_h60,gru,0.417276,0.083942,0.256466,0.379603,0.430914,False
6,t2_p40_h60,transformer,0.410546,0.077213,NaN,NaN,NaN,False
7,t2_p40_h60,xgboost,0.398670,0.065336,0.282040,0.358578,0.437952,False
8,t2_p40_h60,logistic_regression,0.395298,0.061964,0.264022,0.390754,0.439998,False
9,t2_p40_h60,random_forest,0.360983,0.027650,0.280233,0.351420,0.507601,False



Ranking simple por target:


,target,n_configs,n_models,gain_mean,balanced_accuracy_mean,trade_rate_mean,precision_trade_mean,confidence_mean
0,t2_p40_h30,5,5,0.067216,0.400549,0.27325,0.436320,0.434214
1,t2_p50_h30,5,5,0.065933,0.399267,0.23077,0.415675,0.452048
2,t2_p40_h60,5,5,0.063221,0.396555,0.27069,0.370089,0.454116



Estabilidad por target:


,target,gain_mean,gain_std,trade_rate_std,precision_trade_std
0,t2_p40_h30,0.067216,0.012914,0.099144,0.021516
1,t2_p50_h30,0.065933,0.014386,0.024294,0.013141
2,t2_p40_h60,0.063221,0.021775,0.012472,0.018245


## **7.2. Observaciones del análisis por target**

Desempeño predictivo

* `t2_p40_h30` presenta el mayor gain promedio (~0.067), seguido muy de cerca por `t2_p50_h30`.
* `t2_p40_h60` queda en último lugar tanto en gain como en balanced_accuracy.
* Las diferencias entre `p40_h30` y `p50_h30` son pequeñas, lo que indica que ambos capturan señal similar.

Conclusión: `t2_p40_h30` es el target más fuerte en términos predictivos, aunque `t2_p50_h30` es muy competitivo.

---

Métricas operativas

* `t2_p40_h30` tiene el mayor `trade_rate` (~0.27) junto con buena precisión (~0.43).
* `t2_p50_h30` presenta menor frecuencia (~0.23), pero comportamiento más controlado.
* `t2_p40_h60` mantiene frecuencia alta, pero con la peor precisión (~0.37).

Conclusión:

* `p40_h30` es el target más activo y con buena calidad
* `p50_h30` es más selectivo
* `p40_h60` genera señales más débiles

---

Operatividad real

* En `t2_p40_h30` la mayoría de modelos son operables
* En `t2_p50_h30` también hay varios modelos operables
* En `t2_p40_h60` ningún modelo resulta operable

Conclusión clave

`t2_p40_h60` queda descartado desde el punto de vista operativo.

---

Estabilidad

* `t2_p50_h30` es el target más estable (menor std en trade_rate y precision).
* `t2_p40_h30` muestra mayor variabilidad entre modelos (trade_rate_std alto).
* `t2_p40_h60` es el más inestable en términos de señal (gain_std alto).

Conclusión:

* `p50_h30` es el más robusto
* `p40_h30` es más potente pero más variable

---

Lectura global

* `t2_p40_h30` → mejor combinación de señal y actividad
* `t2_p50_h30` → más estable y controlado
* `t2_p40_h60` → menor calidad general y sin operatividad

---

Conclusión final

El análisis muestra que no todos los targets son igualmente útiles.

* `t2_p40_h30` y `t2_p50_h30` concentran la señal real del problema
* `t2_p40_h60` no logra traducir su señal en comportamiento operativo válido

A partir de este punto, el análisis debe centrarse en los targets `p40_h30` y `p50_h30`, descartando `p40_h60` como candidato principal.


# **8. Análisis conjunto (modelo + window_size + target)**


En esta etapa se realiza un análisis combinado de las configuraciones, considerando simultáneamente el modelo y el target. Dado que en el presente experimento se utiliza un único valor de `window_size` (`L = 30`), esta dimensión no introduce variabilidad y se mantiene fija en todas las comparaciones.

A diferencia de los análisis anteriores, donde cada dimensión se evaluó por separado, en este punto se analizan configuraciones específicas para identificar qué combinaciones concretas logran capturar mejor la señal del mercado.

Se construye un ranking de configuraciones en función de:

* métricas de desempeño predictivo (`balanced_accuracy`, `balanced_accuracy_gain_vs_naive`, `f1_macro`)
* métricas operativas (`trade_rate`, `precision_trade`, `confidence`)

Este análisis permite:

* identificar las mejores configuraciones reales (no solo modelos o targets por separado)
* detectar combinaciones que logran un buen equilibrio entre calidad predictiva y utilidad operativa
* comparar setups específicos bajo un mismo criterio experimental

El resultado es un ranking ordenado de combinaciones (modelo + target), que constituye la base directa para la selección final de candidatos a optimizar en etapas posteriores.

Este punto representa el núcleo del proceso de selección, ya que permite pasar de un análisis agregado a decisiones concretas sobre configuraciones específicas.


## **8.2. Código para análisis conjunto**

In [55]:
# ================================
# 8. Análisis conjunto
# model + window_size + target
# ================================

print("8. Análisis conjunto (modelo + window_size + target):\n")

ranking_joint = (
    df_flags
    .groupby(["model", "window_size", "target"], as_index=False)
    .agg(
        balanced_accuracy=("balanced_accuracy", "mean"),
        gain=("balanced_accuracy_gain_vs_naive", "mean"),
        f1_macro=("f1_macro", "mean"),
        trade_rate=("trade_rate", "mean"),
        precision_trade=("precision_trade", "mean"),
        mean_confidence=("mean_confidence", "mean"),
        is_operable=("is_operable", "max"),
        is_strong=("is_strong", "max"),
        is_very_strong=("is_very_strong", "max"),
    )
    .sort_values(
        by=["is_operable", "gain", "precision_trade", "trade_rate"],
        ascending=False
    )
    .reset_index(drop=True)
)

display(ranking_joint)

8. Análisis conjunto (modelo + window_size + target):



,model,window_size,target,balanced_accuracy,gain,f1_macro,trade_rate,precision_trade,mean_confidence,is_operable,is_strong,is_very_strong
0,gru,30,t2_p40_h30,0.415750,0.082417,0.381825,0.180035,0.455206,0.414847,True,True,True
1,xgboost,30,t2_p50_h30,0.408889,0.075556,0.403778,0.259808,0.406040,0.430655,True,True,True
2,xgboost,30,t2_p40_h30,0.405681,0.072348,0.385198,0.259954,0.434880,0.421458,True,True,True
3,logistic_regression,30,t2_p50_h30,0.402972,0.069639,0.388331,0.234525,0.420074,0.434151,True,True,True
4,gru,30,t2_p50_h30,0.402873,0.069539,0.385423,0.200668,0.432295,0.425129,True,True,True
5,logistic_regression,30,t2_p40_h30,0.396396,0.063063,0.366836,0.240046,0.448547,0.429001,True,True,True
6,random_forest,30,t2_p40_h30,0.381023,0.047690,0.365310,0.412965,0.406646,0.471550,True,True,False
7,random_forest,30,t2_p50_h30,0.373990,0.040657,0.316082,0.228080,0.404289,0.518256,True,True,False
8,gru,30,t2_p40_h60,0.417276,0.083942,0.410322,0.256466,0.379603,0.430914,False,True,True
9,transformer,30,t2_p40_h60,0.410546,0.077213,0.407382,NaN,NaN,NaN,False,True,True


## **8.3. Observaciones de análisis conjunto**

Ranking general

* La mejor configuración es `gru + t2_p40_h30`, con el mayor gain (~0.082) y alta precisión (~0.455), aunque con menor frecuencia de operación.
* `xgboost` aparece inmediatamente después en dos combinaciones (`p50_h30` y `p40_h30`), mostrando gran consistencia.
* `logistic_regression` se mantiene competitivo, especialmente en `t2_p50_h30`.

Conclusión: las combinaciones líderes son `gru` y `xgboost`, con ligera ventaja de `gru` en calidad de señal.

---

Targets dominantes

* Las mejores configuraciones corresponden a `t2_p40_h30` y `t2_p50_h30`.
* `t2_p40_h60` aparece en posiciones inferiores o no operables.

Conclusión:

* `p40_h30` es el target más fuerte
* `p50_h30` es una alternativa sólida
* `p40_h60` queda relegado

---

Operatividad

* Las configuraciones operables (`is_operable = True`) coinciden con las primeras posiciones del ranking.
* Las configuraciones con `transformer` no son operables (NaNs en métricas operativas).
* `random_forest` es operable, pero con menor calidad de señal.

Conclusión:

* la operatividad filtra naturalmente los mejores modelos
* no todas las buenas métricas predictivas se traducen en uso real

---

Comparación entre modelos

* `gru`: mejor calidad de señal, menor frecuencia
* `xgboost`: mejor equilibrio entre señal y actividad
* `logistic_regression`: estable y competitivo
* `random_forest`: alta frecuencia pero menor calidad
* `transformer`: buen desempeño predictivo pero no usable operativamente

---

Trade-off clave

* configuraciones con mayor gain (`gru`) tienden a menor `trade_rate`
* configuraciones más activas (`xgboost`, `random_forest`) reducen precisión

Conclusión: existe un trade-off claro entre calidad de señal y frecuencia de operación.

---

Conclusión final

* Las mejores combinaciones reales son:

  * `gru + t2_p40_h30`
  * `xgboost + t2_p50_h30`
  * `xgboost + t2_p40_h30`

* `t2_p40_h30` se consolida como el target principal

* `t2_p50_h30` funciona como alternativa más balanceada

A partir de este punto, ya es posible seleccionar un conjunto reducido de candidatos para validación final o tuning.


#**9. Selección final de candidatos**


En esta etapa se selecciona un conjunto reducido de configuraciones candidatas, evitando elegir un único modelo de forma prematura. El objetivo es conservar diversidad controlada entre alternativas que ya demostraron ser competitivas, permitiendo una comparación más robusta en etapas posteriores.

La selección se basa en tres criterios principales:

- desempeño predictivo (gain y balanced_accuracy)
- estabilidad (consistencia entre targets y baja variabilidad)
- equilibrio entre frecuencia de señal (trade_rate) y calidad (precision_trade)

Se priorizan configuraciones que combinan buen rendimiento global con comportamiento operativo sólido, evitando tanto modelos excesivamente conservadores como aquellos que operan mucho pero con baja calidad.

## **9.1. Código para selección de candidatos**

In [60]:
print("9. Selección de candidatos:\n")

# =========================================
# 0. Base
# =========================================
df_candidates = ranking_joint.copy()

# Filtrar solo operables
df_candidates = df_candidates[
    df_candidates["is_operable"]
].copy()

# =========================================
# 1. Score CON trade_rate
# =========================================
df_candidates["score_with_trade"] = (
    df_candidates["gain"] * 0.6 +
    df_candidates["precision_trade"] * 0.3 +
    df_candidates["trade_rate"] * 0.1
)

df_with_trade = (
    df_candidates
    .sort_values(by="score_with_trade", ascending=False)
    .reset_index(drop=True)
)

# =========================================
# 2. Score SIN trade_rate
# =========================================
df_candidates["score_no_trade"] = (
    df_candidates["gain"] * 0.7 +
    df_candidates["precision_trade"] * 0.3
)

df_no_trade = (
    df_candidates
    .sort_values(by="score_no_trade", ascending=False)
    .reset_index(drop=True)
)

# =========================================
# 3. Top N
# =========================================
top_n = 5

top_with_trade = df_with_trade.head(top_n)
top_no_trade = df_no_trade.head(top_n)

print(f"Top {top_n} CON trade_rate:")
display(top_with_trade)

print(f"\nTop {top_n} SIN trade_rate:")
display(top_no_trade)

# =========================================
# 4. Vista comparativa simple
# =========================================
print("\nComparación simplificada:\n")

display(
    top_with_trade[
        [
            "model",
            "target",
            "gain",
            "trade_rate",
            "precision_trade",
            "score_with_trade",
        ]
    ]
)

display(
    top_no_trade[
        [
            "model",
            "target",
            "gain",
            "trade_rate",
            "precision_trade",
            "score_no_trade",
        ]
    ]
)

9. Selección de candidatos:

Top 5 CON trade_rate:


,model,window_size,target,balanced_accuracy,gain,f1_macro,trade_rate,precision_trade,mean_confidence,is_operable,is_strong,is_very_strong,score_with_trade
0,gru,30,t2_p40_h30,0.415750,0.082417,0.381825,0.180035,0.455206,0.414847,True,True,True,0.204015
1,xgboost,30,t2_p40_h30,0.405681,0.072348,0.385198,0.259954,0.434880,0.421458,True,True,True,0.199868
2,logistic_regression,30,t2_p40_h30,0.396396,0.063063,0.366836,0.240046,0.448547,0.429001,True,True,True,0.196407
3,xgboost,30,t2_p50_h30,0.408889,0.075556,0.403778,0.259808,0.406040,0.430655,True,True,True,0.193126
4,random_forest,30,t2_p40_h30,0.381023,0.047690,0.365310,0.412965,0.406646,0.471550,True,True,False,0.191904



Top 5 SIN trade_rate:


,model,window_size,target,balanced_accuracy,gain,f1_macro,trade_rate,precision_trade,mean_confidence,is_operable,is_strong,is_very_strong,score_with_trade,score_no_trade
0,gru,30,t2_p40_h30,0.415750,0.082417,0.381825,0.180035,0.455206,0.414847,True,True,True,0.204015,0.194254
1,xgboost,30,t2_p40_h30,0.405681,0.072348,0.385198,0.259954,0.434880,0.421458,True,True,True,0.199868,0.181108
2,logistic_regression,30,t2_p40_h30,0.396396,0.063063,0.366836,0.240046,0.448547,0.429001,True,True,True,0.196407,0.178708
3,gru,30,t2_p50_h30,0.402873,0.069539,0.385423,0.200668,0.432295,0.425129,True,True,True,0.191479,0.178366
4,logistic_regression,30,t2_p50_h30,0.402972,0.069639,0.388331,0.234525,0.420074,0.434151,True,True,True,0.191258,0.174769



Comparación simplificada:



,model,target,gain,trade_rate,precision_trade,score_with_trade
0,gru,t2_p40_h30,0.082417,0.180035,0.455206,0.204015
1,xgboost,t2_p40_h30,0.072348,0.259954,0.434880,0.199868
2,logistic_regression,t2_p40_h30,0.063063,0.240046,0.448547,0.196407
3,xgboost,t2_p50_h30,0.075556,0.259808,0.406040,0.193126
4,random_forest,t2_p40_h30,0.047690,0.412965,0.406646,0.191904


,model,target,gain,trade_rate,precision_trade,score_no_trade
0,gru,t2_p40_h30,0.082417,0.180035,0.455206,0.194254
1,xgboost,t2_p40_h30,0.072348,0.259954,0.434880,0.181108
2,logistic_regression,t2_p40_h30,0.063063,0.240046,0.448547,0.178708
3,gru,t2_p50_h30,0.069539,0.200668,0.432295,0.178366
4,logistic_regression,t2_p50_h30,0.069639,0.234525,0.420074,0.174769


## **9.2. Observaciones de la selección de candidatos**

Comparación general

* El ranking es consistente en ambos enfoques: `gru`, `xgboost` y `logistic_regression` dominan las primeras posiciones.
* `gru + t2_p40_h30` se mantiene como el mejor candidato en ambos casos.
* Las diferencias aparecen en las posiciones intermedias, donde influye el uso de `trade_rate`.

---

Impacto de incluir trade_rate

* Al incluir `trade_rate`, aparece `random_forest` en el top 5.
* Esto se debe a su alta frecuencia de operación (~0.41), no a su calidad predictiva.
* Aun así, queda en última posición del top, lo que confirma que su ventaja es limitada.

Conclusión:
El uso de `trade_rate` favorece modelos más activos, incluso si su señal es más débil.

---

Ranking sin trade_rate

* `random_forest` desaparece completamente del top 5.
* Aparecen combinaciones más consistentes como `gru + t2_p50_h30` y `logistic_regression + t2_p50_h30`.
* El ranking refleja mejor la calidad de señal real (gain + precisión).

Conclusión:
El ranking sin `trade_rate` es más alineado con el desempeño predictivo y la calidad de señal.

---

Estabilidad del top

* Las tres primeras posiciones son idénticas en ambos rankings:

  * `gru + t2_p40_h30`
  * `xgboost + t2_p40_h30`
  * `logistic_regression + t2_p40_h30`

* Esto confirma que estas combinaciones son robustas y no dependen del criterio de scoring.

---

Lectura por target

* `t2_p40_h30` domina claramente el top en ambos enfoques.
* `t2_p50_h30` aparece como segunda alternativa en posiciones inferiores.

Conclusión:
`t2_p40_h30` se consolida como el target principal.

---

Trade-off observado

* `gru` → mayor gain y precisión, menor frecuencia
* `xgboost` → buen balance entre frecuencia y calidad
* `logistic_regression` → alta precisión relativa, desempeño estable
* `random_forest` → alta frecuencia pero menor calidad

---

Conclusión final

* Las mejores configuraciones son estables frente al criterio de scoring.
* `gru + t2_p40_h30` es el candidato más fuerte globalmente.
* `xgboost + t2_p40_h30` es la alternativa más equilibrada.
* `logistic_regression` ofrece una opción simple y consistente.

El uso de `trade_rate` debe manejarse con cuidado, ya que puede introducir sesgos hacia modelos más activos sin mejorar la calidad de la señal.


# **10. Validación operativa**

10. Validación operativa

En esta etapa se evalúa el comportamiento de las configuraciones seleccionadas en condiciones cercanas a la operativa real, utilizando directamente el dataset de probabilidades. El objetivo es validar que el modelo no solo presenta buen desempeño en métricas, sino que también genera señales utilizables en un contexto de trading.

Se aplica una regla de decisión basada en umbrales sobre las probabilidades para generar señales de trading (long/short), evitando decisiones basadas únicamente en `argmax`. A partir de estas señales, se analizan las siguientes métricas operativas:

* número de trades por período (`trade_rate` y conteo total)
* cantidad de señales útiles (`is_correct` condicionado a trades)
* precisión de las señales (`precision_trade`)
* distribución temporal de las señales (por ejemplo, por día)

Este análisis permite detectar comportamientos como:

* modelos que generan demasiadas señales (sobretrading)
* modelos que generan pocas señales (subutilización)
* concentración temporal de señales
* diferencias entre rendimiento global y rendimiento operativo

---

Interpretación

* `n_trades` y `trade_rate`: permiten evaluar la frecuencia de operación
* `precision_trade`: indica la calidad real de las señales ejecutadas
* distribución por día: muestra si la actividad es consistente o concentrada

---

Conclusión

Este paso permite validar que las configuraciones seleccionadas no solo son buenas en métricas, sino que también son viables en un entorno operativo real. Es el puente entre el modelo predictivo y su aplicación en trading.



## **10.1. Código de validación operativa (top candidatos)**

In [61]:
print("10. Validación operativa:\n")

# =========================================
# 1. Seleccionar candidatos
# =========================================
selected = top_no_trade.copy()  # o top_with_trade

# =========================================
# 2. Filtrar df_proba por candidatos
# =========================================
df_eval = df_proba.merge(
    selected[["model", "target"]],
    on=["model", "target"],
    how="inner"
)

# =========================================
# 3. Métricas operativas agregadas
# =========================================
oper_summary = (
    df_eval
    .groupby(["model", "target"], as_index=False)
    .agg(
        n_obs=("trade", "size"),
        n_trades=("trade", "sum"),
        trade_rate=("trade", "mean"),
        precision_trade=("is_correct", lambda x: x[df_eval.loc[x.index, "trade"]].mean()
                         if df_eval.loc[x.index, "trade"].any() else 0.0),
        mean_confidence=("confidence", "mean"),
    )
    .sort_values("precision_trade", ascending=False)
)

print("Resumen operativo:")
display(oper_summary)

# =========================================
# 4. Señales por día (si hay columna date)
# =========================================
if "date" in df_eval.columns:
    print("\nDistribución temporal de señales (por día):")

    trades_by_day = (
        df_eval[df_eval["trade"]]
        .groupby(["model", "target", "date"])
        .size()
        .rename("n_trades")
        .reset_index()
    )

    display(trades_by_day.head())

    print("\nResumen por día:")
    display(
        trades_by_day.groupby(["model", "target"])["n_trades"].describe()
    )

10. Validación operativa:

Resumen operativo:


,model,target,n_obs,n_trades,trade_rate,precision_trade,mean_confidence
0,gru,t2_p40_h30,6882,1239,0.180035,0.455206,0.414847
2,logistic_regression,t2_p40_h30,6882,1652,0.240046,0.448547,0.429001
4,xgboost,t2_p40_h30,6882,1789,0.259954,0.434880,0.421458
1,gru,t2_p50_h30,6882,1381,0.200668,0.432295,0.425129
3,logistic_regression,t2_p50_h30,6882,1614,0.234525,0.420074,0.434151


## **10.2. Observaciones de la validación operativa**



Frecuencia de operación

* `xgboost + t2_p40_h30` es el modelo más activo (~1789 trades, ~0.26).
* `logistic_regression` presenta una frecuencia intermedia (~0.23–0.24).
* `gru` es el más conservador (~0.18–0.20).

Conclusión: existe una clara jerarquía en actividad → `xgboost > logistic > gru`.

---

Calidad de señal

* `gru + t2_p40_h30` tiene la mayor precisión (~0.455), siendo el modelo con mejor calidad de señal.
* `logistic_regression` se mantiene muy cercano (~0.448), mostrando gran solidez.
* `xgboost` tiene menor precisión (~0.435), aunque sigue siendo aceptable.

Conclusión:

* `gru` lidera en calidad
* `logistic` es muy competitivo
* `xgboost` sacrifica precisión por mayor actividad

---

Comparación por target

* `t2_p40_h30` domina en todos los modelos:

  * mayor precisión
  * mejor equilibrio general
* `t2_p50_h30` muestra menor precisión en todos los casos.

Conclusión:
`t2_p40_h30` se confirma como el mejor target también en operativa real.

---

Trade-off clave

* `gru`: menos trades, mayor precisión → enfoque conservador
* `xgboost`: más trades, menor precisión → enfoque agresivo
* `logistic_regression`: punto intermedio → balanceado

Conclusión:
hay un trade-off claro entre frecuencia y calidad de señal.

---

Consistencia con análisis previo

* Los resultados operativos son coherentes con todo lo analizado anteriormente:

  * ranking por modelo
  * análisis de probabilidades
  * selección de candidatos

Esto valida que el proceso de selección fue correcto.

---

Conclusión final

* `gru + t2_p40_h30` es la mejor opción si se prioriza calidad de señal.
* `xgboost + t2_p40_h30` es la mejor opción si se prioriza actividad.
* `logistic_regression + t2_p40_h30` ofrece el mejor equilibrio entre ambos.

El análisis confirma que las configuraciones seleccionadas son viables operativamente y que la elección final depende del perfil de trading que se quiera implementar.


# **11. Definición del espacio de tuning**


Sí, Gus, con eso alcanza.
Ya tenemos una base suficiente para definir el punto 11 de forma concreta.

Lo más razonable es:

* modelos a tunear: `GRU`, `XGBoost`, `Logistic Regression`
* targets a conservar: `t2_p40_h30` y `t2_p50_h30`
* estrategia: tuneo grueso → selección de mejores combinaciones → tuneo fino

Te dejo el texto del punto para la notebook.

11. Definición del espacio de tuning

Una vez identificadas las configuraciones más prometedoras, se define el espacio de optimización para la siguiente etapa del proceso. El objetivo es refinar únicamente los modelos y targets que ya demostraron señal predictiva y utilidad operativa, evitando expandir el costo computacional sobre configuraciones que fueron descartadas en etapas previas.

En esta etapa se seleccionan como candidatos principales los modelos `GRU`, `XGBoost` y `Logistic Regression`, junto con los targets `t2_p40_h30` y `t2_p50_h30`, que fueron los mejor posicionados en el análisis conjunto y en la validación operativa.

Dado que el número de combinaciones ya quedó reducido, se propone una estrategia de optimización en dos niveles:

* tuneo grueso, orientado a explorar regiones amplias del espacio de hiperparámetros
* tuneo fino, orientado a refinar las mejores configuraciones encontradas en la etapa anterior

La optimización se organizará en tres dimensiones principales:

* hiperparámetros del modelo
* umbrales de decisión sobre probabilidades
* posibles mejoras en el conjunto de features

Se toma como punto de partida la configuración base actualmente utilizada en cada modelo:

Logistic Regression

* `max_iter = 1000`
* `C = 1.0`
* `multi_class = "multinomial"`
* `solver = "lbfgs"`
* `class_weight = "balanced"`
* `prob_threshold_long = 0.40`
* `prob_threshold_short = 0.40`

GRU

* `hidden_size = 128`
* `num_layers = 1`
* `dropout = 0.1`
* `learning_rate = 1e-3`
* `batch_size = 2048`
* `eval_batch_size = 2048`
* `epochs = 20`
* `patience = 5`
* `optimizer_name = "adamw"`
* `grad_clip_norm = 1.0`
* `class_weight = "balanced"`

XGBoost

* `n_estimators = 200`
* `max_depth = 3`
* `learning_rate = 0.03`
* `subsample = 0.8`
* `colsample_bytree = 0.8`
* `min_child_weight = 1`
* `gamma = 0.0`
* `reg_alpha = 0.0`
* `reg_lambda = 10.0`
* `class_weight = "balanced"`
* `tree_method = "hist"`
* `prob_threshold_long = 0.40`
* `prob_threshold_short = 0.40`

A partir de estas configuraciones base, el tuning se planteará de forma progresiva, priorizando primero ajustes gruesos en hiperparámetros estructurales y luego ajustes más finos en regularización, thresholds y selección de variables.

Esta estrategia permite equilibrar costo computacional y calidad de búsqueda, y establece una base ordenada para la próxima etapa experimental.


## **11.1. Definición de tuneo grueso vs fino**

El objetivo es separar:

* qué parámetros cambian la forma del modelo (tuneo grueso)
* cuáles ajustan detalles finos del comportamiento (tuneo fino)

---

**Logistic Regression**

- Tuneo grueso

  * `C` → controla la regularización (parámetro clave)

    * valores a probar: `[0.01, 0.1, 1, 10, 100]`

- Tuneo fino

  * `C` en un rango más acotado alrededor del mejor valor encontrado
  * `prob_threshold_long`, `prob_threshold_short`

    * valores a probar: `[0.35, 0.40, 0.45, 0.50]`

---

**GRU**

- Tuneo grueso

  * `hidden_size` → capacidad del modelo

    * `[64, 128, 256]`
  * `num_layers`

    * `[1, 2]`
  * `learning_rate`

    * `[1e-4, 5e-4, 1e-3]`

- Tuneo fino

  * `dropout` → `[0.0, 0.1, 0.2, 0.3]`
  * `batch_size` → `[1024, 2048, 4096]`
  * `grad_clip_norm` → `[0.5, 1.0, 2.0]`
  * thresholds de probabilidad

---

**XGBoost**

- Tuneo grueso

  * `max_depth` → `[3, 5, 7]`
  * `learning_rate` → `[0.01, 0.03, 0.1]`
  * `n_estimators` → `[200, 400, 600]`

- Tuneo fino

  * `subsample` → `[0.6, 0.8, 1.0]`
  * `colsample_bytree` → `[0.6, 0.8, 1.0]`
  * `gamma` → `[0, 0.1, 0.3]`
  * `reg_alpha` → `[0, 0.1, 1]`
  * `reg_lambda` → `[1, 10, 50]`
  * thresholds de probabilidad

---

Muy importante (clave conceptual)

* Los thresholds (`prob_threshold_*`) no forman parte del tuning del modelo
* Deben ajustarse posteriormente como parte de la estrategia operativa

---

Orden correcto de trabajo

1. Tuneo grueso (modelo)
2. Selección de mejores configuraciones
3. Tuneo fino (modelo)
4. Ajuste de thresholds (operativa)

---

**Conclusión**

* La separación entre tuneo grueso y fino permite estructurar el proceso de optimización
* El espacio de búsqueda queda definido de forma controlada
* Es posible avanzar al tuning sin necesidad de redefinir criterios en etapas posteriores


# **Resumen**

La estructura del análisis desarrollada es consistente y adecuada, permitiendo evaluar de forma integral el desempeño de los modelos desde una perspectiva tanto predictiva como operativa.

El proceso se refuerza mediante tres elementos clave:

* definición explícita de métricas de decisión, estableciendo criterios claros para comparar modelos en términos de desempeño y mejora respecto al baseline
* incorporación del análisis de probabilidades, que permite entender el comportamiento real de los modelos más allá de métricas agregadas
* validación operativa, conectando el rendimiento del modelo con su aplicación práctica en generación de señales de trading

A partir de este enfoque, se logró:

* identificar los modelos más robustos (`GRU`, `XGBoost`, `Logistic Regression`)
* determinar los targets más relevantes (`t2_p40_h30` y `t2_p50_h30`)
* evidenciar el trade-off entre calidad de señal y frecuencia de operación
* construir un ranking consistente de configuraciones reales
* validar que las configuraciones seleccionadas son operativamente viables

El resultado es un conjunto reducido de candidatos de alta calidad, junto con un espacio de tuning bien definido, que permite avanzar de forma estructurada hacia la optimización de modelos sin incurrir en exploraciones innecesarias o poco informadas.


# **Conclusión final**

Se mantiene la estructura general del análisis, incorporando ajustes que permiten evaluar los modelos no solo desde el punto de vista predictivo, sino también como sistemas de generación de señales para trading.

El enfoque adoptado permitió pasar de una evaluación basada únicamente en métricas agregadas a un proceso integral que considera desempeño, comportamiento probabilístico y utilidad operativa. Esto posibilitó identificar no solo qué modelos predicen mejor, sino cuáles generan señales efectivas en un entorno cercano a la operativa real.

Como resultado, se definió un conjunto claro de modelos (`GRU`, `XGBoost` y `Logistic Regression`) y targets (`t2_p40_h30`, `t2_p50_h30`) que concentran la señal relevante del problema, junto con una estrategia de optimización estructurada basada en tuneo progresivo.

Este proceso establece una base sólida para la siguiente etapa, donde el foco estará en refinar los modelos seleccionados mediante tuning y ajustes operativos, con el objetivo de maximizar la calidad y consistencia de las señales generadas.
